In [1]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client

db = DuckDBClient()
s  = S3Client()

# Load all clean USDINR data (both contracts)
all_files  = s.list_files('processed/1s/USDINR26SEPFUT/')
all_files += s.list_files('processed/1s/USDINR26OCTFUT/')

all_df = []
for f in all_files:
    parts = f.split('/')
    sym   = parts[2]
    date  = parts[3].replace('.parquet','')
    try:
        df = db.read_parquet(f'processed/1s/{sym}/{date}.parquet')
        df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
        df = df[(df['ist_sec'] >= 9*3600) &
                (df['ist_sec'] <= 17*3600)].reset_index(drop=True)
        df['date']   = date
        df['symbol'] = sym
        all_df.append(df)
        print(f'{sym}/{date}: {len(df)} bars')
    except Exception as e:
        print(f'ERROR {sym}/{date}: {e}')

db.close()
df_raw = pd.concat(all_df, ignore_index=True).reset_index(drop=True)
print(f'\nTotal: {len(df_raw):,} bars across {df_raw["date"].nunique()} days')
print(f'Dates: {sorted(df_raw["date"].unique())}')
print(f'Columns: {len(df_raw.columns)}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26SEPFUT/2026-09-22: 12568 bars
USDINR26SEPFUT/2026-09-23: 11031 bars
USDINR26SEPFUT/2026-09-24: 10385 bars
USDINR26SEPFUT/2026-09-25: 15469 bars
USDINR26SEPFUT/2026-09-28: 2921 bars
USDINR26OCTFUT/2026-09-30: 16701 bars
USDINR26OCTFUT/2026-10-01: 16554 bars
USDINR26OCTFUT/2026-10-05: 14944 bars
USDINR26OCTFUT/2026-10-06: 18290 bars
USDINR26OCTFUT/2026-10-07: 19336 bars
USDINR26OCTFUT/2026-10-08: 20968 bars
USDINR26OCTFUT/2026-10-09: 20206 bars
USDINR26OCTFUT/2026-10-12: 18406 bars
USDINR26OCTFUT/2026-10-13: 18435 bars
USDINR26OCTFUT/2026-10-14: 17290 bars

Total: 233,504 bars across 15 days
Dates: ['2026-09-22', '2026-09-23', '2026-09-24', '2026-09-25', '2026-09-28', '2026-09-30', '2026-10-01', '2026-10-05', '2026-10-06', '2026-10-07', '2026-10-08', '2026-10-09', '2026-10-12', '2026-10-13', '2026-10-14']
Columns: 55


In [3]:
df_raw.columns

Index(['symbol', 'ts_sec', 'open', 'high', 'low', 'close', 'volume_delta',
       'tick_count', 'vwap', 'oi', 'spread_mean', 'spread_max', 'spread_bps',
       'imbalance_mean', 'imbalance_std', 'imbalance_last', 'total_bid_qty',
       'total_ask_qty', 'weighted_mid', 'price_impact', 'bid_p1', 'bid_q1',
       'bid_p2', 'bid_q2', 'bid_p3', 'bid_q3', 'bid_p4', 'bid_q4', 'bid_p5',
       'bid_q5', 'ask_p1', 'ask_q1', 'ask_p2', 'ask_q2', 'ask_p3', 'ask_q3',
       'ask_p4', 'ask_q4', 'ask_p5', 'ask_q5', 'realized_vol_10s',
       'realized_vol_30s', 'realized_vol_60s', 'realized_vol_300s',
       'imbalance_ma_10s', 'imbalance_ma_30s', 'imbalance_ma_60s',
       'spread_zscore', 'volume_ratio', 'price_mom_10s', 'price_mom_30s',
       'price_mom_60s', 'ts_ist', 'ist_sec', 'date'],
      dtype='object')

In [4]:
features = df_raw.copy()

In [5]:
df = df_raw.sort_values(["symbol", "ts_sec"]).reset_index(drop=True)

print(df.shape)
print(df["date"].value_counts().sort_index())

print("Duplicate timestamps:",
      df.duplicated(["symbol", "ts_sec"]).sum())

print("\nMissing values:")
print(
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .head(20)
)

(233504, 55)
date
2026-09-22    12568
2026-09-23    11031
2026-09-24    10385
2026-09-25    15469
2026-09-28     2921
2026-09-30    16701
2026-10-01    16554
2026-10-05    14944
2026-10-06    18290
2026-10-07    19336
2026-10-08    20968
2026-10-09    20206
2026-10-12    18406
2026-10-13    18435
2026-10-14    17290
Name: count, dtype: int64
Duplicate timestamps: 0

Missing values:
imbalance_std        0.881505
price_mom_60s        0.003854
price_mom_30s        0.001927
price_mom_10s        0.000642
spread_zscore        0.000475
realized_vol_60s     0.000064
realized_vol_300s    0.000064
realized_vol_10s     0.000064
realized_vol_30s     0.000064
oi                   0.000000
vwap                 0.000000
spread_mean          0.000000
spread_max           0.000000
spread_bps           0.000000
imbalance_mean       0.000000
volume_delta         0.000000
tick_count           0.000000
ts_sec               0.000000
symbol               0.000000
high                 0.000000
dtype: float64


In [6]:
df["dt"] = (
    df.groupby("symbol")["ts_sec"]
      .diff()
)

print(df["dt"].value_counts().head(10))

dt
1.0     186593
2.0      20583
3.0       9576
4.0       5205
5.0       3600
6.0       1762
7.0       1265
8.0        944
9.0        692
10.0       516
Name: count, dtype: int64


In [7]:
import numpy as np

eps = 1e-12

features["mid"] = (
    features["bid_p1"] + features["ask_p1"]
) / 2

features["spread"] = (
    features["ask_p1"] - features["bid_p1"]
)

features["spread_bps_calc"] = (
    1e4 * features["spread"] /
    (features["mid"] + eps)
)

In [8]:
print(
    features[
        ["mid", "spread", "spread_bps_calc",
         "spread_mean", "spread_bps"]
    ].head()
)

        mid  spread  spread_bps_calc  spread_mean  spread_bps
0  95.75125  0.0075         0.783280       0.0075    0.783280
1  95.75125  0.0075         0.783280       0.0075    0.783280
2  95.75125  0.0075         0.783280       0.0075    0.783280
3  95.75125  0.0075         0.783280       0.0075    0.783280
4  95.75750  0.0050         0.522152       0.0050    0.522152


In [9]:
features["mid_log"] = np.log(features["mid"])

features["ret_1s"] = (
    features
    .groupby(["symbol", "date"])["mid_log"]
    .diff()
)

In [10]:
bid_q_cols = [f"bid_q{i}" for i in range(1, 6)]
ask_q_cols = [f"ask_q{i}" for i in range(1, 6)]

features["bid_depth_calc"] = features[bid_q_cols].sum(axis=1)
features["ask_depth_calc"] = features[ask_q_cols].sum(axis=1)

features["total_depth"] = (
    features["bid_depth_calc"] +
    features["ask_depth_calc"]
)

In [11]:
print(
    features[
        [
            "bid_depth_calc",
            "total_bid_qty",
            "ask_depth_calc",
            "total_ask_qty"
        ]
    ].head()
)

   bid_depth_calc  total_bid_qty  ask_depth_calc  total_ask_qty
0            2610          24616             642           6773
1            2410          24416             642           6773
2            2610          24416             642           6773
3            1810          24616             642           6373
4            1820          24626             910           6241


In [12]:
features["depth_imbalance"] = (
    (
        features["bid_depth_calc"] -
        features["ask_depth_calc"]
    )
    /
    (
        features["total_depth"] + eps
    )
)

In [13]:
features["queue_imbalance_l1"] = (
    (features["bid_q1"] - features["ask_q1"])
    /
    (features["bid_q1"] + features["ask_q1"] + eps)
)

In [14]:
features["microprice"] = (
    features["ask_p1"] * features["bid_q1"]
    +
    features["bid_p1"] * features["ask_q1"]
) / (
    features["bid_q1"] +
    features["ask_q1"] +
    eps
)

In [15]:
features["microprice_displacement"] = (
    (features["microprice"] - features["mid"])
    /
    (features["spread"] + eps)
)

In [18]:
def add_snapshot_ofi(df, level):
    bp = f"bid_p{level}"
    bq = f"bid_q{level}"
    ap = f"ask_p{level}"
    aq = f"ask_q{level}"

    prev_bp = df.groupby(["symbol", "date"])[bp].shift(1)
    prev_bq = df.groupby(["symbol", "date"])[bq].shift(1)

    prev_ap = df.groupby(["symbol", "date"])[ap].shift(1)
    prev_aq = df.groupby(["symbol", "date"])[aq].shift(1)

    bid_event = (
        (df[bp] >= prev_bp).astype(float) * df[bq]
        -
        (df[bp] <= prev_bp).astype(float) * prev_bq
    )

    ask_event = (
        -
        (df[ap] <= prev_ap).astype(float) * df[aq]
        +
        (df[ap] >= prev_ap).astype(float) * prev_aq
    )

    df[f"ofi_l{level}"] = bid_event + ask_event

    return df
for level in range(1, 6):
    features = add_snapshot_ofi(features, level)

In [19]:
features["avg_depth"] = (
    features["bid_depth_calc"] +
    features["ask_depth_calc"]
) / 2

for level in range(1, 6):
    features[f"nofi_l{level}"] = (
        features[f"ofi_l{level}"]
        /
        (features["avg_depth"] + eps)
    )

In [20]:
g = features.groupby(["symbol", "date"])

features["nofi_l1_delta"] = (
    g["nofi_l1"].diff()
)

features["nofi_l1_accel"] = (
    features
    .groupby(["symbol", "date"])["nofi_l1_delta"]
    .diff()
)

In [21]:
cols = [
    "ret_1s",
    "spread_bps_calc",
    "total_depth",
    "depth_imbalance",
    "queue_imbalance_l1",
    "microprice_displacement",
    "ofi_l1",
    "nofi_l1",
    "nofi_l1_delta",
    "nofi_l1_accel",
]

print(features[cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5,
                 0.75, 0.95, 0.99]
).T)

                            count          mean          std           min  \
ret_1s                   233489.0 -2.792111e-09     0.000022     -0.002208   
spread_bps_calc          233504.0  9.728848e-01     0.692837      0.257357   
total_depth              233504.0  4.875109e+03  3961.426790    233.000000   
depth_imbalance          233504.0  1.577674e-01     0.414335     -0.951124   
queue_imbalance_l1       233504.0  8.115248e-02     0.676151     -0.999334   
microprice_displacement  233504.0  4.057624e-02     0.338075     -0.499667   
ofi_l1                   233489.0  1.785129e+00   291.538182 -13737.000000   
nofi_l1                  233489.0  1.821177e-03     0.148024     -4.727316   
nofi_l1_delta            233474.0 -6.208052e-06     0.216893     -8.130212   
nofi_l1_accel            233459.0  5.925794e-07     0.379898    -15.230184   

                                  1%           5%          25%          50%  \
ret_1s                     -0.000065    -0.000013     0.000000

In [22]:
print(
    features[cols]
    .replace([np.inf, -np.inf], np.nan)
    .isna()
    .mean()
    .sort_values(ascending=False)
)

nofi_l1_accel              0.000193
nofi_l1_delta              0.000128
ofi_l1                     0.000064
ret_1s                     0.000064
nofi_l1                    0.000064
spread_bps_calc            0.000000
microprice_displacement    0.000000
queue_imbalance_l1         0.000000
depth_imbalance            0.000000
total_depth                0.000000
dtype: float64


In [23]:
daily_time = (
    df.groupby("date")
      .agg(
          first_ts=("ts_ist", "min"),
          last_ts=("ts_ist", "max"),
          rows=("ts_ist", "size")
      )
)

print(daily_time)

                       first_ts              last_ts   rows
date                                                       
2026-09-22  2026-09-22 09:26:33  2026-09-22 16:59:27  12568
2026-09-23  2026-09-23 09:03:55  2026-09-23 16:57:51  11031
2026-09-24  2026-09-24 09:02:36  2026-09-24 16:59:22  10385
2026-09-25  2026-09-25 09:05:55  2026-09-25 16:59:54  15469
2026-09-28  2026-09-28 09:55:14  2026-09-28 12:29:54   2921
2026-09-30  2026-09-30 09:00:00  2026-09-30 16:59:08  16701
2026-10-01  2026-10-01 09:06:20  2026-10-01 16:59:41  16554
2026-10-05  2026-10-05 09:00:00  2026-10-05 16:59:26  14944
2026-10-06  2026-10-06 09:00:00  2026-10-06 16:59:14  18290
2026-10-07  2026-10-07 09:00:00  2026-10-07 16:59:11  19336
2026-10-08  2026-10-08 09:00:00  2026-10-08 16:59:08  20968
2026-10-09  2026-10-09 09:00:00  2026-10-09 16:59:07  20206
2026-10-12  2026-10-12 09:00:00  2026-10-12 16:59:13  18406
2026-10-13  2026-10-13 09:00:00  2026-10-13 16:59:52  18435
2026-10-14  2026-10-14 09:00:00  2026-10

In [24]:
features.loc[
    features["spread_bps_calc"] >
    features["spread_bps_calc"].quantile(0.999),
    [
        "date", "ts_ist",
        "bid_p1", "ask_p1",
        "spread_bps_calc",
        "bid_q1", "ask_q1"
    ]
].head(30)

,date,ts_ist,bid_p1,ask_p1,spread_bps_calc,bid_q1,ask_q1
330,2026-09-22,2026-09-22 09:39:52,95.7500,95.8000,5.220569,398,928
78280,2026-10-01,2026-10-01 13:46:05,96.4300,96.4950,6.738370,220,50
78281,2026-10-01,2026-10-01 13:46:06,96.4300,96.4950,6.738370,200,50
78282,2026-10-01,2026-10-01 13:46:07,96.4300,96.4950,6.738370,200,50
78283,2026-10-01,2026-10-01 13:46:08,96.4300,96.4950,6.738370,200,50
78284,2026-10-01,2026-10-01 13:46:09,96.4300,96.4800,5.183764,200,10
78285,2026-10-01,2026-10-01 13:46:10,96.4300,96.4800,5.183764,200,10
78289,2026-10-01,2026-10-01 13:46:14,96.4350,96.4900,5.701698,100,575
78290,2026-10-01,2026-10-01 13:46:15,96.4350,96.4900,5.701698,100,575
78291,2026-10-01,2026-10-01 13:46:16,96.4350,96.4900,5.701698,100,575


In [25]:
diff = (
    features["microprice_displacement"]
    - 0.5 * features["queue_imbalance_l1"]
)

print(diff.abs().describe())

count    2.335040e+05
mean     1.016062e-10
std      2.835287e-10
min      0.000000e+00
25%      2.187377e-11
50%      4.858841e-11
75%      9.896373e-11
max      1.916192e-08
dtype: float64


### Building 1s grid

In [28]:
import pandas as pd
import numpy as np

# Safety copy
raw = df.copy()

raw["ts_ist"] = pd.to_datetime(raw["ts_ist"])
raw = raw.sort_values(["symbol", "date", "ts_ist"]).reset_index(drop=True)

# Mark genuinely observed rows before reindexing
raw["is_observed"] = 1

In [29]:
book_state_cols = [
    "bid_p1", "bid_q1",
    "bid_p2", "bid_q2",
    "bid_p3", "bid_q3",
    "bid_p4", "bid_q4",
    "bid_p5", "bid_q5",
    "ask_p1", "ask_q1",
    "ask_p2", "ask_q2",
    "ask_p3", "ask_q3",
    "ask_p4", "ask_q4",
    "ask_p5", "ask_q5",
    "oi",
]

In [30]:
flow_cols = [
    "volume_delta",
    "tick_count",
]

In [31]:
derived_cols = [
    "open", "high", "low", "close",
    "vwap",
    "spread_mean", "spread_max", "spread_bps",
    "imbalance_mean", "imbalance_std", "imbalance_last",
    "total_bid_qty", "total_ask_qty",
    "weighted_mid", "price_impact",
    "realized_vol_10s", "realized_vol_30s",
    "realized_vol_60s", "realized_vol_300s",
    "imbalance_ma_10s", "imbalance_ma_30s",
    "imbalance_ma_60s",
    "spread_zscore",
    "volume_ratio",
    "price_mom_10s", "price_mom_30s",
    "price_mom_60s",
]

In [32]:
def regularize_group(g):
    g = g.sort_values("ts_ist").copy()

    symbol = g["symbol"].iloc[0]
    date = g["date"].iloc[0]

    start = g["ts_ist"].min()
    end = g["ts_ist"].max()

    full_index = pd.date_range(
        start=start,
        end=end,
        freq="1s"
    )

    g = (
        g.set_index("ts_ist")
         .reindex(full_index)
    )

    g.index.name = "ts_ist"

    # Restore identifiers
    g["symbol"] = symbol
    g["date"] = date

    # 1 = real source observation, 0 = inserted second
    g["is_observed"] = g["is_observed"].fillna(0).astype(int)

    # Book / state variables persist
    g[book_state_cols] = g[book_state_cols].ffill()

    # No activity in an inserted second
    g[flow_cols] = g[flow_cols].fillna(0)

    return g.reset_index()

In [33]:
parts = []

for (_, _), g in raw.groupby(["symbol", "date"], sort=False):
    parts.append(regularize_group(g))

df_1s = pd.concat(parts, ignore_index=True)

df_1s = df_1s.sort_values(
    ["symbol", "date", "ts_ist"]
).reset_index(drop=True)

In [34]:
df_1s["dt"] = (
    df_1s
    .groupby(["symbol", "date"])["ts_ist"]
    .diff()
    .dt.total_seconds()
)

print(df_1s["dt"].value_counts().head())

dt
1.0    409186
Name: count, dtype: int64


In [35]:
print("Old rows:", len(df))
print("Regularized rows:", len(df_1s))

print(
    df_1s["is_observed"]
    .value_counts(normalize=True)
)

Old rows: 233504
Regularized rows: 409201
is_observed
1    0.570634
0    0.429366
Name: proportion, dtype: float64


In [36]:
df_1s["had_update"] = df_1s["is_observed"]

In [37]:
df_1s["update_rate_30s"] = (
    df_1s
    .groupby(["symbol", "date"])["had_update"]
    .transform(
        lambda x: x.rolling(30, min_periods=1).mean()
    )
)

In [38]:
eps = 1e-12

df_1s["mid"] = (
    df_1s["bid_p1"] +
    df_1s["ask_p1"]
) / 2

df_1s["spread"] = (
    df_1s["ask_p1"] -
    df_1s["bid_p1"]
)

df_1s["spread_bps_calc"] = (
    1e4 *
    df_1s["spread"] /
    (df_1s["mid"] + eps)
)

In [39]:
df_1s["log_mid"] = np.log(df_1s["mid"])

df_1s["ret_1s"] = (
    df_1s
    .groupby(["symbol", "date"])["log_mid"]
    .diff()
)

In [40]:
bid_q_cols = [f"bid_q{i}" for i in range(1, 6)]
ask_q_cols = [f"ask_q{i}" for i in range(1, 6)]

df_1s["bid_depth_l5"] = df_1s[bid_q_cols].sum(axis=1)

df_1s["ask_depth_l5"] = df_1s[ask_q_cols].sum(axis=1)

df_1s["total_depth_l5"] = (
    df_1s["bid_depth_l5"] +
    df_1s["ask_depth_l5"]
)

In [41]:
df_1s["depth_imbalance_l5"] = (
    df_1s["bid_depth_l5"] -
    df_1s["ask_depth_l5"]
) / (
    df_1s["total_depth_l5"] + eps
)

In [42]:
df_1s["queue_imbalance_l1"] = (
    df_1s["bid_q1"] -
    df_1s["ask_q1"]
) / (
    df_1s["bid_q1"] +
    df_1s["ask_q1"] +
    eps
)

In [43]:
def add_snapshot_ofi(df, level):
    group = df.groupby(["symbol", "date"])

    bp = f"bid_p{level}"
    bq = f"bid_q{level}"
    ap = f"ask_p{level}"
    aq = f"ask_q{level}"

    prev_bp = group[bp].shift(1)
    prev_bq = group[bq].shift(1)

    prev_ap = group[ap].shift(1)
    prev_aq = group[aq].shift(1)

    bid_of = (
        (df[bp] >= prev_bp).astype(float) * df[bq]
        -
        (df[bp] <= prev_bp).astype(float) * prev_bq
    )

    ask_of = (
        -(df[ap] <= prev_ap).astype(float) * df[aq]
        +
        (df[ap] >= prev_ap).astype(float) * prev_aq
    )

    df[f"ofi_l{level}"] = bid_of + ask_of

    return df

In [44]:
for level in range(1, 6):
    df_1s = add_snapshot_ofi(df_1s, level)

In [45]:
for level in range(1, 6):

    local_depth = (
        df_1s[f"bid_q{level}"] +
        df_1s[f"ask_q{level}"]
    ) / 2

    df_1s[f"nofi_l{level}"] = (
        df_1s[f"ofi_l{level}"]
        /
        (local_depth + eps)
    )

In [46]:
check_cols = [
    "ret_1s",
    "spread_bps_calc",
    "bid_depth_l5",
    "ask_depth_l5",
    "total_depth_l5",
    "depth_imbalance_l5",
    "queue_imbalance_l1",
    "ofi_l1",
    "ofi_l2",
    "ofi_l3",
    "ofi_l4",
    "ofi_l5",
    "nofi_l1",
    "nofi_l2",
    "nofi_l3",
    "nofi_l4",
    "nofi_l5",
    "update_rate_30s",
]

In [47]:
print(
    df_1s[check_cols]
    .describe(
        percentiles=[
            0.001, 0.01, 0.05,
            0.25, 0.50, 0.75,
            0.95, 0.99, 0.999
        ]
    )
    .T
)

                       count          mean          std           min  \
ret_1s              409186.0 -1.593229e-09     0.000017     -0.002208   
spread_bps_calc     409201.0  9.561153e-01     0.640279      0.257357   
bid_depth_l5        409201.0  2.997931e+03  3511.181707     27.000000   
ask_depth_l5        409201.0  1.679347e+03  1291.688698     17.000000   
total_depth_l5      409201.0  4.677278e+03  3837.870676    233.000000   
depth_imbalance_l5  409201.0  1.688157e-01     0.413986     -0.951124   
queue_imbalance_l1  409201.0  9.359615e-02     0.677972     -0.999334   
ofi_l1              409186.0  1.018627e+00   220.227408 -13737.000000   
ofi_l2              409186.0  2.454972e-01   290.946589 -14083.000000   
ofi_l3              409186.0  1.065779e+00   366.962921 -20451.000000   
ofi_l4              409186.0 -1.323017e-01   383.929877 -22170.000000   
ofi_l5              409186.0  7.421613e-01   437.209997 -20051.000000   
nofi_l1             409186.0  1.556461e-02     3.01

In [48]:
for c in ["ofi_l1", "nofi_l1", "ret_1s"]:
    zero_pct = (df_1s[c] == 0).mean()

    print(
        c,
        "zero percentage:",
        round(100 * zero_pct, 2),
        "%"
    )

ofi_l1 zero percentage: 83.76 %
nofi_l1 zero percentage: 83.76 %
ret_1s zero percentage: 92.95 %


In [49]:
print(
    df_1s.groupby("date").agg(
        rows=("ts_ist", "size"),
        observed=("is_observed", "sum"),
        observation_rate=("is_observed", "mean")
    )
)

             rows  observed  observation_rate
date                                         
2026-09-22  27175     12568          0.462484
2026-09-23  28437     11031          0.387910
2026-09-24  28607     10385          0.363023
2026-09-25  28440     15469          0.543917
2026-09-28   9281      2921          0.314729
2026-09-30  28749     16701          0.580925
2026-10-01  28402     16554          0.582846
2026-10-05  28767     14944          0.519484
2026-10-06  28755     18290          0.636063
2026-10-07  28752     19336          0.672510
2026-10-08  28749     20968          0.729347
2026-10-09  28748     20206          0.702866
2026-10-12  28754     18406          0.640120
2026-10-13  28793     18435          0.640260
2026-10-14  28792     17290          0.600514


In [50]:
df_1s["ofi_l1_active"] = (df_1s["ofi_l1"] != 0).astype(int)

df_1s["ofi_l1_sign"] = np.sign(df_1s["ofi_l1"])

df_1s["ofi_l1_logmag"] = np.log1p(
    np.abs(df_1s["ofi_l1"])
)

df_1s["ofi_l1_signed_log"] = (
    np.sign(df_1s["ofi_l1"])
    * np.log1p(np.abs(df_1s["ofi_l1"]))
)

In [51]:
g = df_1s.groupby(["symbol", "date"])

l1_depth = (
    df_1s["bid_q1"] + df_1s["ask_q1"]
) / 2

df_1s["l1_depth"] = l1_depth

df_1s["l1_depth_med_300s"] = (
    g["l1_depth"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=60
        ).median()
    )
)

df_1s["nofi_l1_robust"] = (
    df_1s["ofi_l1"]
    /
    (df_1s["l1_depth_med_300s"] + 1e-12)
)

In [52]:
df_1s["update_rate_300s"] = (
    df_1s
    .groupby(["symbol", "date"])["had_update"]
    .transform(
        lambda x: x.rolling(
            300,
            min_periods=1
        ).mean()
    )
)

In [53]:
for h in [10, 30, 60, 300]:
    df_1s[f"ofi_l1_sum_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["ofi_l1"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=max(1, h // 3)
            ).sum()
        )
    )

In [54]:
for h in [10, 30, 60, 300]:
    c = f"ofi_l1_sum_{h}s"
    df_1s[f"{c}_signed_log"] = (
        np.sign(df_1s[c])
        * np.log1p(np.abs(df_1s[c]))
    )

In [55]:
def rolling_robust_z(s, window=1800, min_periods=300):
    med = s.rolling(
        window,
        min_periods=min_periods
    ).median()

    mad = (
        (s - med)
        .abs()
        .rolling(
            window,
            min_periods=min_periods
        )
        .median()
    )

    return (s - med) / (1.4826 * mad + 1e-12)

In [56]:
for h in [10, 30, 60, 300]:

    src = f"ofi_l1_sum_{h}s_signed_log"

    df_1s[f"ofi_l1_z_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])[src]
        .transform(
            lambda x: rolling_robust_z(
                x,
                window=1800,
                min_periods=300
            )
        )
    )

In [57]:
df_1s["ofi_term_30_300"] = (
    df_1s["ofi_l1_z_30s"]
    -
    df_1s["ofi_l1_z_300s"]
)

In [58]:
df_1s["price_move_flag"] = (
    df_1s["ret_1s"] != 0
).astype(int)

In [59]:
df_1s["price_move_rate_30s"] = (
    df_1s
    .groupby(["symbol", "date"])["price_move_flag"]
    .transform(
        lambda x: x.rolling(30, min_periods=1).mean()
    )
)

In [60]:
next_cols = [
    "ofi_l1_signed_log",
    "nofi_l1_robust",
    "ofi_l1_sum_10s_signed_log",
    "ofi_l1_sum_30s_signed_log",
    "ofi_l1_sum_60s_signed_log",
    "ofi_l1_sum_300s_signed_log",
    "ofi_l1_z_30s",
    "ofi_l1_z_300s",
    "ofi_term_30_300",
    "update_rate_30s",
    "update_rate_300s",
    "price_move_rate_30s",
]

In [61]:
print(
    df_1s[next_cols]
    .describe(
        percentiles=[
            0.001, 0.01, 0.05,
            0.25, 0.5, 0.75,
            0.95, 0.99, 0.999
        ]
    )
    .T
)

                               count          mean           std  \
ofi_l1_signed_log           409186.0  2.941983e-03  2.041256e+00   
nofi_l1_robust              408316.0  6.039626e-03  1.261096e+00   
ofi_l1_sum_10s_signed_log   409156.0  9.605433e-02  4.050539e+00   
ofi_l1_sum_30s_signed_log   409051.0  2.571039e-01  5.187632e+00   
ofi_l1_sum_60s_signed_log   408901.0  4.249427e-01  5.855535e+00   
ofi_l1_sum_300s_signed_log  407701.0  1.013252e+00  7.111103e+00   
ofi_l1_z_30s                400081.0  4.503424e+09  4.935309e+11   
ofi_l1_z_300s               398731.0 -4.422977e-01  3.605840e+00   
ofi_term_30_300             398731.0  3.744521e+09  4.862331e+11   
update_rate_30s             409201.0  5.707705e-01  3.231222e-01   
update_rate_300s            409201.0  5.724685e-01  2.433755e-01   
price_move_rate_30s         409201.0  7.073024e-02  1.039046e-01   

                                     min          0.1%         1%        5%  \
ofi_l1_signed_log          -9.527921

In [62]:
print(
    df_1s[next_cols]
    .replace([np.inf, -np.inf], np.nan)
    .isna()
    .mean()
    .sort_values(ascending=False)
)

ofi_term_30_300               0.025586
ofi_l1_z_300s                 0.025586
ofi_l1_z_30s                  0.022287
ofi_l1_sum_300s_signed_log    0.003666
nofi_l1_robust                0.002163
ofi_l1_sum_60s_signed_log     0.000733
ofi_l1_sum_30s_signed_log     0.000367
ofi_l1_sum_10s_signed_log     0.000110
ofi_l1_signed_log             0.000037
update_rate_30s               0.000000
update_rate_300s              0.000000
price_move_rate_30s           0.000000
dtype: float64


In [63]:
for h in [10, 30, 60, 300]:
    df_1s[f"ofi_l1_rate_{h}s"] = (
        df_1s[f"ofi_l1_sum_{h}s"] / h
    )

In [64]:
df_1s["ofi_rate_diff_30_300"] = (
    df_1s["ofi_l1_rate_30s"]
    -
    df_1s["ofi_l1_rate_300s"]
)

df_1s["ofi_rate_diff_30_300_sl"] = (
    np.sign(df_1s["ofi_rate_diff_30_300"])
    *
    np.log1p(
        np.abs(df_1s["ofi_rate_diff_30_300"])
    )
)

In [65]:
for h in [30, 60, 300]:

    pos = (
        (df_1s["ofi_l1"] > 0)
        .astype(int)
        .groupby([df_1s["symbol"], df_1s["date"]])
        .transform(
            lambda x: x.rolling(h, min_periods=h//3).sum()
        )
    )

    neg = (
        (df_1s["ofi_l1"] < 0)
        .astype(int)
        .groupby([df_1s["symbol"], df_1s["date"]])
        .transform(
            lambda x: x.rolling(h, min_periods=h//3).sum()
        )
    )

    df_1s[f"ofi_dir_consistency_{h}s"] = (
        (pos - neg) /
        (pos + neg + 1e-12)
    )

In [66]:
df_1s["ofi_active"] = (df_1s["ofi_l1"] != 0).astype(int)

for h in [30, 60, 300]:
    df_1s[f"ofi_activity_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["ofi_active"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=h//3
            ).mean()
        )
    )

In [67]:
df_1s["nofi_l1_robust_sl"] = (
    np.sign(df_1s["nofi_l1_robust"])
    *
    np.log1p(
        np.abs(df_1s["nofi_l1_robust"])
    )
)

In [68]:
bad_cols = [
    "ofi_l1_z_30s",
    "ofi_l1_z_300s",
    "ofi_term_30_300",
]

df_1s = df_1s.drop(
    columns=[c for c in bad_cols if c in df_1s.columns]
)

In [70]:
horizons = [30, 60, 300, 900, 1800]

g = df_1s.groupby(["symbol", "date"])

for h in horizons:
    df_1s[f"fwd_ret_{h}s"] = (
        g["log_mid"].shift(-h)
        - df_1s["log_mid"]
    )

In [71]:
ofi_features = [
    "ofi_l1_signed_log",
    "nofi_l1_robust_sl",

    "ofi_l1_sum_10s_signed_log",
    "ofi_l1_sum_30s_signed_log",
    "ofi_l1_sum_60s_signed_log",
    "ofi_l1_sum_300s_signed_log",

    "ofi_rate_diff_30_300_sl",

    "ofi_dir_consistency_30s",
    "ofi_dir_consistency_60s",
    "ofi_dir_consistency_300s",

    "ofi_activity_30s",
    "ofi_activity_60s",
    "ofi_activity_300s",
]

In [72]:
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

results = []

for feature in ofi_features:

    for h in horizons:

        target = f"fwd_ret_{h}s"

        temp = df_1s[[feature, target]].dropna()

        ic, pval = spearmanr(
            temp[feature],
            temp[target]
        )

        results.append({
            "feature": feature,
            "horizon": h,
            "spearman_ic": ic,
            "pvalue_naive": pval
        })

ic_table = pd.DataFrame(results)

print(
    ic_table
    .sort_values(
        "spearman_ic",
        key=lambda x: x.abs(),
        ascending=False
    )
)

                       feature  horizon  spearman_ic   pvalue_naive
48    ofi_dir_consistency_300s      900     0.042510  4.393524e-157
64           ofi_activity_300s     1800    -0.037233  7.136741e-117
21   ofi_l1_sum_60s_signed_log       60     0.033611  2.659900e-102
63           ofi_activity_300s      900    -0.033072   7.989628e-96
53            ofi_activity_30s      900    -0.032234   1.990213e-91
..                         ...      ...          ...            ...
1            ofi_l1_signed_log       60     0.002333   1.360640e-01
6            nofi_l1_robust_sl       60     0.002044   1.920366e-01
0            ofi_l1_signed_log       30     0.001866   2.329399e-01
5            nofi_l1_robust_sl       30     0.001775   2.568740e-01
29  ofi_l1_sum_300s_signed_log     1800     0.001552   3.382857e-01

[65 rows x 4 columns]


In [73]:
daily_results = []

for date, day in df_1s.groupby("date"):

    for feature in ofi_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            daily_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

daily_ic = pd.DataFrame(daily_results)

In [74]:
ic_summary = (
    daily_ic
    .groupby(["feature", "horizon"])
    .agg(
        mean_daily_ic=("ic", "mean"),
        median_daily_ic=("ic", "median"),
        std_daily_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

ic_summary["ic_stability"] = (
    ic_summary["mean_daily_ic"]
    /
    (ic_summary["std_daily_ic"] + 1e-12)
)

print(
    ic_summary.sort_values(
        "ic_stability",
        key=lambda x: x.abs(),
        ascending=False
    )
)

                      feature  horizon  mean_daily_ic  median_daily_ic  \
10           ofi_activity_30s       30      -0.026492        -0.017559   
11           ofi_activity_30s       60      -0.030737        -0.017489   
15           ofi_activity_60s       30      -0.022130        -0.014684   
16           ofi_activity_60s       60      -0.025853        -0.023820   
56  ofi_l1_sum_60s_signed_log       60       0.025153         0.032115   
..                        ...      ...            ...              ...   
35          ofi_l1_signed_log       30       0.000400        -0.000104   
1           nofi_l1_robust_sl       60       0.000330         0.000673   
30    ofi_dir_consistency_60s       30       0.001234        -0.003265   
44  ofi_l1_sum_10s_signed_log     1800      -0.000565         0.004243   
63    ofi_rate_diff_30_300_sl      900       0.000414        -0.004500   

    std_daily_ic  positive_days  n_days  ic_stability  
10      0.032983       0.133333      15     -0.803199  

In [75]:
mean_ic_pivot = ic_summary.pivot(
    index="feature",
    columns="horizon",
    values="mean_daily_ic"
)

print(mean_ic_pivot.round(4))

horizon                       30      60      300     900     1800
feature                                                           
nofi_l1_robust_sl           0.0004  0.0003  0.0029  0.0030  0.0012
ofi_activity_300s          -0.0133 -0.0176 -0.0366 -0.0455 -0.0427
ofi_activity_30s           -0.0265 -0.0307 -0.0361 -0.0450 -0.0356
ofi_activity_60s           -0.0221 -0.0259 -0.0340 -0.0447 -0.0353
ofi_dir_consistency_300s    0.0047  0.0093  0.0072  0.0371  0.0102
ofi_dir_consistency_30s    -0.0053  0.0019  0.0087  0.0079  0.0052
ofi_dir_consistency_60s     0.0012  0.0103  0.0104  0.0095  0.0039
ofi_l1_signed_log           0.0004  0.0005  0.0032  0.0031  0.0013
ofi_l1_sum_10s_signed_log   0.0050  0.0043  0.0060  0.0072 -0.0006
ofi_l1_sum_300s_signed_log  0.0115  0.0124 -0.0065  0.0190 -0.0244
ofi_l1_sum_30s_signed_log   0.0065  0.0131  0.0112  0.0111 -0.0043
ofi_l1_sum_60s_signed_log   0.0161  0.0252  0.0137  0.0159 -0.0066
ofi_rate_diff_30_300_sl     0.0094  0.0140  0.0186  0.0004  0.

In [76]:
positive_pivot = ic_summary.pivot(
    index="feature",
    columns="horizon",
    values="positive_days"
)

print(positive_pivot.round(2))

horizon                     30    60    300   900   1800
feature                                                 
nofi_l1_robust_sl           0.47  0.53  0.53  0.47  0.60
ofi_activity_300s           0.27  0.27  0.33  0.47  0.40
ofi_activity_30s            0.13  0.20  0.27  0.27  0.33
ofi_activity_60s            0.20  0.27  0.33  0.33  0.33
ofi_dir_consistency_300s    0.53  0.60  0.60  0.60  0.47
ofi_dir_consistency_30s     0.47  0.47  0.47  0.60  0.53
ofi_dir_consistency_60s     0.47  0.67  0.53  0.53  0.47
ofi_l1_signed_log           0.47  0.53  0.53  0.60  0.60
ofi_l1_sum_10s_signed_log   0.60  0.53  0.67  0.67  0.53
ofi_l1_sum_300s_signed_log  0.67  0.67  0.53  0.60  0.40
ofi_l1_sum_30s_signed_log   0.67  0.67  0.60  0.60  0.53
ofi_l1_sum_60s_signed_log   0.67  0.73  0.67  0.67  0.53
ofi_rate_diff_30_300_sl     0.67  0.73  0.73  0.40  0.60


In [77]:
median_ic_pivot = ic_summary.pivot(
    index="feature",
    columns="horizon",
    values="median_daily_ic"
)

print(median_ic_pivot.round(4))

horizon                       30      60      300     900     1800
feature                                                           
nofi_l1_robust_sl          -0.0003  0.0007  0.0023 -0.0001  0.0031
ofi_activity_300s          -0.0098 -0.0101 -0.0382 -0.0069 -0.0505
ofi_activity_30s           -0.0176 -0.0175 -0.0436 -0.0339 -0.0410
ofi_activity_60s           -0.0147 -0.0238 -0.0576 -0.0385 -0.0461
ofi_dir_consistency_300s    0.0002  0.0120  0.0211  0.0190 -0.0135
ofi_dir_consistency_30s    -0.0127 -0.0028 -0.0027  0.0073  0.0009
ofi_dir_consistency_60s    -0.0033  0.0025  0.0059  0.0054 -0.0058
ofi_l1_signed_log          -0.0001  0.0016  0.0023  0.0002  0.0034
ofi_l1_sum_10s_signed_log   0.0100  0.0098  0.0092  0.0087  0.0042
ofi_l1_sum_300s_signed_log  0.0163  0.0173  0.0038  0.0465 -0.0225
ofi_l1_sum_30s_signed_log   0.0263  0.0238  0.0223  0.0112  0.0003
ofi_l1_sum_60s_signed_log   0.0160  0.0321  0.0218  0.0115  0.0030
ofi_rate_diff_30_300_sl     0.0201  0.0224  0.0282 -0.0045  0.

In [78]:
for h in horizons:
    df_1s[f"abs_fwd_ret_{h}s"] = (
        df_1s[f"fwd_ret_{h}s"].abs()
    )

In [79]:
activity_features = [
    "ofi_activity_30s",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_30s",
    "update_rate_300s",
    "price_move_rate_30s",
]

In [80]:
for h in [30, 60, 300, 900, 1800]:
    df_1s[f"abs_fwd_ret_{h}s"] = (
        df_1s[f"fwd_ret_{h}s"].abs()
    )

In [81]:
activity_features = [
    "ofi_activity_30s",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_30s",
    "update_rate_300s",
    "price_move_rate_30s",
]

In [82]:
activity_results = []

for date, day in df_1s.groupby("date"):

    for feature in activity_features:

        for h in [30, 60, 300, 900, 1800]:

            target = f"abs_fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            activity_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

activity_daily = pd.DataFrame(activity_results)

In [83]:
activity_summary = (
    activity_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

print(
    activity_summary
    .pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    )
    .round(4)
)

print(
    activity_summary
    .pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    )
    .round(2)
)

horizon                30      60      300     900     1800
feature                                                    
ofi_activity_300s    0.2007  0.2120  0.1700  0.1681  0.1131
ofi_activity_30s     0.2460  0.1889  0.1385  0.0987  0.0872
ofi_activity_60s     0.2211  0.1814  0.1490  0.1121  0.0975
price_move_rate_30s  0.2623  0.2010  0.1182  0.0752  0.0725
update_rate_300s     0.1340  0.1485  0.1460  0.1684  0.1218
update_rate_30s      0.1822  0.1373  0.1231  0.1068  0.1010
horizon              30    60    300   900   1800
feature                                          
ofi_activity_300s    1.00  1.00  0.87  0.93  0.53
ofi_activity_30s     1.00  1.00  1.00  0.87  0.53
ofi_activity_60s     1.00  0.87  1.00  0.87  0.53
price_move_rate_30s  1.00  1.00  0.93  0.73  0.60
update_rate_300s     0.93  1.00  0.93  0.80  0.67
update_rate_30s      1.00  0.93  1.00  0.80  0.67


In [86]:
df_1s["ret_30s"] = (
    df_1s["log_mid"]
    -
    df_1s
    .groupby(["symbol", "date"])["log_mid"]
    .shift(30)
)

In [87]:
xcol = "ofi_l1_sum_30s_signed_log"
ycol = "ret_30s"

window = 1800
min_obs = 600

In [88]:
df_1s["_impact_xy"] = df_1s[xcol] * df_1s[ycol]
df_1s["_impact_x2"] = df_1s[xcol] ** 2
df_1s["_impact_y2"] = df_1s[ycol] ** 2

In [89]:
g = df_1s.groupby(["symbol", "date"])

In [90]:
roll_x = g[xcol].transform(
    lambda x: x.rolling(window, min_periods=min_obs).mean()
)

roll_y = g[ycol].transform(
    lambda x: x.rolling(window, min_periods=min_obs).mean()
)

roll_xy = g["_impact_xy"].transform(
    lambda x: x.rolling(window, min_periods=min_obs).mean()
)

roll_x2 = g["_impact_x2"].transform(
    lambda x: x.rolling(window, min_periods=min_obs).mean()
)

roll_y2 = g["_impact_y2"].transform(
    lambda x: x.rolling(window, min_periods=min_obs).mean()
)

In [91]:
cov_xy = roll_xy - roll_x * roll_y
var_x = roll_x2 - roll_x**2
var_y = roll_y2 - roll_y**2

In [92]:
eps = 1e-12

df_1s["impact_lambda_1800s"] = (
    cov_xy / (var_x + eps)
)

In [93]:
df_1s["impact_alpha_1800s"] = (
    roll_y
    -
    df_1s["impact_lambda_1800s"] * roll_x
)

In [94]:
rolling_corr = (
    cov_xy /
    np.sqrt((var_x + eps) * (var_y + eps))
)

df_1s["impact_r2_1800s"] = rolling_corr**2

In [95]:
df_1s["impact_expected_ret"] = (
    df_1s["impact_alpha_1800s"]
    +
    df_1s["impact_lambda_1800s"]
    * df_1s[xcol]
)

In [96]:
df_1s["impact_residual"] = (
    df_1s[ycol]
    -
    df_1s["impact_expected_ret"]
)

In [97]:
df_1s["absorption_raw"] = (
    -np.sign(df_1s[xcol])
    * df_1s["impact_residual"]
)

In [98]:
df_1s["impact_resid_vol"] = (
    df_1s
    .groupby(["symbol", "date"])["impact_residual"]
    .transform(
        lambda x: x.rolling(
            1800,
            min_periods=600
        ).std()
    )
)

In [99]:
df_1s["absorption_z"] = (
    df_1s["absorption_raw"]
    /
    (df_1s["impact_resid_vol"] + eps)
)

In [100]:
for c in [
    "impact_lambda_1800s",
    "impact_alpha_1800s",
    "impact_r2_1800s"
]:
    df_1s[c] = (
        df_1s
        .groupby(["symbol", "date"])[c]
        .shift(1)
    )

In [101]:
df_1s["impact_expected_ret"] = (
    df_1s["impact_alpha_1800s"]
    +
    df_1s["impact_lambda_1800s"]
    * df_1s[xcol]
)

df_1s["impact_residual"] = (
    df_1s[ycol]
    -
    df_1s["impact_expected_ret"]
)

In [102]:
impact_cols = [
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "impact_residual",
    "absorption_z"
]

print(
    df_1s[impact_cols]
    .describe(
        percentiles=[
            .001, .01, .05,
            .25, .5, .75,
            .95, .99, .999
        ]
    ).T
)

print(
    df_1s[impact_cols]
    .replace([np.inf, -np.inf], np.nan)
    .isna()
    .mean()
)

                        count          mean       std           min      0.1%  \
impact_lambda_1800s  399751.0  5.772731e-06  0.000003  8.417288e-07  0.000001   
impact_r2_1800s      399751.0  3.131112e-01  0.098284  2.575808e-02  0.054403   
impact_residual      399751.0  5.092817e-08  0.000051 -1.435461e-03 -0.000376   
absorption_z         390781.0  2.535246e-02  1.048957 -1.811215e+01 -6.655129   

                           1%        5%       25%           50%       75%  \
impact_lambda_1800s  0.000002  0.000003  0.000004  5.231412e-06  0.000007   
impact_r2_1800s      0.082991  0.137663  0.247906  3.182210e-01  0.383855   
impact_residual     -0.000154 -0.000071 -0.000017  4.426062e-07  0.000019   
absorption_z        -3.498083 -1.809471 -0.202461  0.000000e+00  0.527219   

                          95%       99%     99.9%        max  
impact_lambda_1800s  0.000012  0.000015  0.000017   0.000022  
impact_r2_1800s      0.467197  0.517506  0.586128   0.618668  
impact_residual    

In [103]:
signal_col = "ofi_l1_sum_60s_signed_log"

In [104]:
q_low = df_1s[signal_col].quantile(0.20)
q_high = df_1s[signal_col].quantile(0.80)

df_1s["strong_buy_ofi"] = (
    df_1s[signal_col] >= q_high
).astype(int)

df_1s["strong_sell_ofi"] = (
    df_1s[signal_col] <= q_low
).astype(int)

In [105]:
df_1s["high_absorption"] = (
    df_1s["absorption_z"] >= 1.0
).astype(int)

df_1s["low_absorption"] = (
    df_1s["absorption_z"] <= -1.0
).astype(int)

In [106]:
horizons = [30, 60, 300, 900, 1800]

rows = []

for h in horizons:
    target = f"fwd_ret_{h}s"

    conditions = {
        "buy_high_abs": (
            (df_1s["strong_buy_ofi"] == 1) &
            (df_1s["high_absorption"] == 1)
        ),
        "buy_low_abs": (
            (df_1s["strong_buy_ofi"] == 1) &
            (df_1s["low_absorption"] == 1)
        ),
        "sell_high_abs": (
            (df_1s["strong_sell_ofi"] == 1) &
            (df_1s["high_absorption"] == 1)
        ),
        "sell_low_abs": (
            (df_1s["strong_sell_ofi"] == 1) &
            (df_1s["low_absorption"] == 1)
        ),
    }

    for name, mask in conditions.items():

        vals = df_1s.loc[mask, target].dropna()

        rows.append({
            "horizon": h,
            "condition": name,
            "n": len(vals),
            "mean_ret": vals.mean(),
            "median_ret": vals.median(),
            "positive_frac": (vals > 0).mean()
        })

conditional_results = pd.DataFrame(rows)

print(
    conditional_results.pivot(
        index="condition",
        columns="horizon",
        values="mean_ret"
    )
)

print(
    conditional_results.pivot(
        index="condition",
        columns="horizon",
        values="positive_frac"
    )
)

print(
    conditional_results.pivot(
        index="condition",
        columns="horizon",
        values="n"
    )
)

horizon            30        60        300       900       1800
condition                                                      
buy_high_abs   0.000010  0.000011  0.000012  0.000010  0.000016
buy_low_abs   -0.000010 -0.000010 -0.000003  0.000031  0.000066
sell_high_abs -0.000009 -0.000011 -0.000017 -0.000022  0.000024
sell_low_abs  -0.000006 -0.000009 -0.000014 -0.000007  0.000025
horizon            30        60        300       900       1800
condition                                                      
buy_high_abs   0.427239  0.483573  0.542587  0.549515  0.539007
buy_low_abs    0.292325  0.353985  0.463349  0.521384  0.544874
sell_high_abs  0.304906  0.353670  0.467154  0.521527  0.584826
sell_low_abs   0.428603  0.465116  0.539760  0.554462  0.584229
horizon         30     60     300    900    1800
condition                                       
buy_high_abs    9112   9101   9099   8977   8883
buy_low_abs    12965  12961  12960  12743  12524
sell_high_abs   8806   8802   8753  

In [107]:
ofi_sign_60 = np.sign(
    df_1s["ofi_l1_sum_60s_signed_log"]
)

for h in [30, 60, 300, 900, 1800]:
    df_1s[f"continuation_ret_{h}s"] = (
        ofi_sign_60 *
        df_1s[f"fwd_ret_{h}s"]
    )

In [108]:
strong_ofi = (
    (df_1s["strong_buy_ofi"] == 1) |
    (df_1s["strong_sell_ofi"] == 1)
)

In [109]:
rows = []

for h in [30, 60, 300, 900, 1800]:

    target = f"continuation_ret_{h}s"

    for name, mask in {
        "high_absorption": (
            strong_ofi &
            (df_1s["absorption_z"] >= 1)
        ),
        "neutral_absorption": (
            strong_ofi &
            (df_1s["absorption_z"].between(-1, 1))
        ),
        "low_absorption": (
            strong_ofi &
            (df_1s["absorption_z"] <= -1)
        )
    }.items():

        x = df_1s.loc[mask, target].dropna()

        rows.append({
            "horizon": h,
            "regime": name,
            "n": len(x),
            "mean_cont": x.mean(),
            "median_cont": x.median(),
            "continuation_frac": (x > 0).mean(),
            "reversal_frac": (x < 0).mean()
        })

cont_results = pd.DataFrame(rows)

print(
    cont_results.pivot(
        index="regime",
        columns="horizon",
        values="mean_cont"
    )
)

print(
    cont_results.pivot(
        index="regime",
        columns="horizon",
        values="continuation_frac"
    )
)

print(
    cont_results.pivot(
        index="regime",
        columns="horizon",
        values="reversal_frac"
    )
)

horizon                 30            60        300       900       1800
regime                                                                  
high_absorption     0.000009  1.098114e-05  0.000014  0.000016 -0.000003
low_absorption     -0.000002 -5.403947e-07  0.000006  0.000018  0.000020
neutral_absorption  0.000004  6.996411e-06  0.000008  0.000014  0.000013
horizon                 30        60        300       900       1800
regime                                                              
high_absorption     0.410369  0.466682  0.506498  0.497556  0.472849
low_absorption      0.319417  0.368174  0.438508  0.468791  0.474613
neutral_absorption  0.331489  0.396521  0.481286  0.500019  0.498946
horizon                 30        60        300       900       1800
regime                                                              
high_absorption     0.275087  0.321343  0.428635  0.470444  0.506221
low_absorption      0.438206  0.474020  0.514856  0.496417  0.510388
neutral_absorp

In [110]:
daily_abs_results = []

for date, day in df_1s.groupby("date"):

    strong = (
        (day["strong_buy_ofi"] == 1) |
        (day["strong_sell_ofi"] == 1)
    )

    for h in [30, 60, 300, 900, 1800]:

        target = f"continuation_ret_{h}s"

        high = day.loc[
            strong & (day["absorption_z"] >= 1),
            target
        ].dropna()

        low = day.loc[
            strong & (day["absorption_z"] <= -1),
            target
        ].dropna()

        if len(high) < 50 or len(low) < 50:
            continue

        daily_abs_results.append({
            "date": date,
            "horizon": h,
            "high_mean": high.mean(),
            "low_mean": low.mean(),
            "difference": high.mean() - low.mean()
        })

daily_abs = pd.DataFrame(daily_abs_results)

In [111]:
print(
    daily_abs.groupby("horizon").agg(
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=("difference", lambda x: (x > 0).mean()),
        n_days=("difference", "count")
    )
)

         mean_difference  median_difference  positive_days  n_days
horizon                                                           
30              0.000013           0.000015       0.866667      15
60              0.000014           0.000018       0.866667      15
300             0.000010           0.000013       0.600000      15
900            -0.000002          -0.000004       0.466667      15
1800           -0.000020          -0.000025       0.466667      15


In [112]:
ofi_cols = [f"ofi_l{i}" for i in range(1, 6)]

corr = df_1s[ofi_cols].corr(method="spearman")
print(corr.round(3))

        ofi_l1  ofi_l2  ofi_l3  ofi_l4  ofi_l5
ofi_l1   1.000   0.307   0.259   0.292   0.315
ofi_l2   0.307   1.000   0.304   0.324   0.346
ofi_l3   0.259   0.304   1.000   0.438   0.460
ofi_l4   0.292   0.324   0.438   1.000   0.735
ofi_l5   0.315   0.346   0.460   0.735   1.000


In [113]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

tmp = df_1s[ofi_cols].dropna()

X = StandardScaler().fit_transform(tmp)

pca = PCA()
Z = pca.fit_transform(X)

print("Explained variance ratio:")
print(pca.explained_variance_ratio_)

print("\nCumulative:")
print(np.cumsum(pca.explained_variance_ratio_))

print("\nComponents:")
components = pd.DataFrame(
    pca.components_,
    columns=ofi_cols,
    index=[f"PC{i}" for i in range(1, 6)]
)

print(components.round(3))

Explained variance ratio:
[0.442619   0.18349342 0.14294001 0.12294047 0.1080071 ]

Cumulative:
[0.442619   0.62611242 0.76905243 0.8919929  1.        ]

Components:
     ofi_l1  ofi_l2  ofi_l3  ofi_l4  ofi_l5
PC1   0.343   0.440   0.493   0.494   0.449
PC2   0.736   0.345  -0.046  -0.358  -0.456
PC3  -0.577   0.631   0.308  -0.101  -0.406
PC4   0.081  -0.534   0.690   0.147  -0.458
PC5   0.032   0.063  -0.428   0.772  -0.465


In [114]:
for level in range(1, 6):

    c = f"ofi_l{level}"

    df_1s[f"ofi_l{level}_sl"] = (
        np.sign(df_1s[c])
        *
        np.log1p(np.abs(df_1s[c]))
    )

In [115]:
ofi_sl_cols = [
    f"ofi_l{i}_sl"
    for i in range(1, 6)
]

tmp_sl = df_1s[ofi_sl_cols].dropna()

scaler_sl = StandardScaler()
X_sl = scaler_sl.fit_transform(tmp_sl)

pca_sl = PCA()
Z_sl = pca_sl.fit_transform(X_sl)

print("Explained variance:")
print(pca_sl.explained_variance_ratio_)

print("\nCumulative:")
print(np.cumsum(pca_sl.explained_variance_ratio_))

components_sl = pd.DataFrame(
    pca_sl.components_,
    columns=ofi_sl_cols,
    index=[f"PC{i}" for i in range(1, 6)]
)

print("\nComponents:")
print(components_sl.round(3))

Explained variance:
[0.5062898  0.16995584 0.13553515 0.12612804 0.06209117]

Cumulative:
[0.5062898  0.67624564 0.81178079 0.93790883 1.        ]

Components:
     ofi_l1_sl  ofi_l2_sl  ofi_l3_sl  ofi_l4_sl  ofi_l5_sl
PC1      0.369      0.386      0.436      0.507      0.517
PC2      0.641      0.541     -0.158     -0.389     -0.347
PC3     -0.661      0.683      0.255     -0.135     -0.121
PC4      0.128     -0.303      0.849     -0.303     -0.283
PC5     -0.017     -0.021     -0.013     -0.694      0.720


In [116]:
print(
    df_1s[ofi_sl_cols]
    .corr(method="spearman")
    .round(3)
)

           ofi_l1_sl  ofi_l2_sl  ofi_l3_sl  ofi_l4_sl  ofi_l5_sl
ofi_l1_sl      1.000      0.307      0.259      0.292      0.315
ofi_l2_sl      0.307      1.000      0.304      0.324      0.346
ofi_l3_sl      0.259      0.304      1.000      0.438      0.460
ofi_l4_sl      0.292      0.324      0.438      1.000      0.735
ofi_l5_sl      0.315      0.346      0.460      0.735      1.000


In [117]:
valid_idx = tmp_sl.index

df_1s.loc[valid_idx, "ofi_pc1"] = Z_sl[:, 0]
df_1s.loc[valid_idx, "ofi_pc2"] = Z_sl[:, 1]

In [118]:
df_1s["ofi_book_sum_sl"] = (
    df_1s[[f"ofi_l{i}_sl" for i in range(1, 6)]]
    .sum(axis=1)
)

In [119]:
print(
    df_1s[["ofi_pc1", "ofi_book_sum_sl"]]
    .corr()
)

                  ofi_pc1  ofi_book_sum_sl
ofi_pc1          1.000000         0.993942
ofi_book_sum_sl  0.993942         1.000000


In [120]:
if df_1s[["ofi_pc1", "ofi_book_sum_sl"]].corr().iloc[0, 1] < 0:
    df_1s["ofi_pc1"] *= -1

In [121]:
df_1s["ofi_shape_raw"] = (
    (
        df_1s["ofi_l1_sl"] +
        df_1s["ofi_l2_sl"]
    ) / 2
    -
    (
        df_1s["ofi_l4_sl"] +
        df_1s["ofi_l5_sl"]
    ) / 2
)

In [122]:
if df_1s[["ofi_pc2", "ofi_shape_raw"]].corr().iloc[0, 1] < 0:
    df_1s["ofi_pc2"] *= -1

In [123]:
pc_features = [
    "ofi_pc1",
    "ofi_pc2",
    "ofi_l1_sum_30s_signed_log",
    "ofi_l1_sum_60s_signed_log"
]

In [124]:
pc_features = [
    "ofi_pc1",
    "ofi_pc2"
]

horizons = [30, 60, 300, 900, 1800]

pc_daily_results = []

for date, day in df_1s.groupby("date"):

    for feature in pc_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            pc_daily_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

pc_daily = pd.DataFrame(pc_daily_results)

In [125]:
pc_summary = (
    pc_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_daily_ic=("ic", "mean"),
        median_daily_ic=("ic", "median"),
        std_daily_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

print(
    pc_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_daily_ic"
    ).round(4)
)

print(
    pc_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon    30      60      300     900     1800
feature                                        
ofi_pc1  0.0023  0.0025  0.0033  0.0028  0.0009
ofi_pc2  0.0005 -0.0011 -0.0004 -0.0011 -0.0010
horizon  30    60    300   900   1800
feature                              
ofi_pc1  0.60  0.67  0.67   0.6  0.60
ofi_pc2  0.47  0.60  0.40   0.4  0.33


In [126]:
for pc in ["ofi_pc1", "ofi_pc2"]:
    for h in [10, 30, 60, 300]:

        df_1s[f"{pc}_sum_{h}s"] = (
            df_1s
            .groupby(["symbol", "date"])[pc]
            .transform(
                lambda x: x.rolling(
                    h,
                    min_periods=max(1, h // 3)
                ).sum()
            )
        )

In [127]:
agg_pc_features = [
    "ofi_pc1_sum_10s",
    "ofi_pc1_sum_30s",
    "ofi_pc1_sum_60s",
    "ofi_pc1_sum_300s",
    "ofi_pc2_sum_10s",
    "ofi_pc2_sum_30s",
    "ofi_pc2_sum_60s",
    "ofi_pc2_sum_300s",
]

In [128]:
agg_results = []

for date, day in df_1s.groupby("date"):

    for feature in agg_pc_features:

        for h in [30, 60, 300, 900, 1800]:

            target = f"fwd_ret_{h}s"
            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            agg_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

agg_daily = pd.DataFrame(agg_results)

agg_summary = (
    agg_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean())
    )
    .reset_index()
)

print(
    agg_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    agg_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon             30      60      300     900     1800
feature                                                 
ofi_pc1_sum_10s  -0.0045 -0.0038  0.0012  0.0060 -0.0013
ofi_pc1_sum_300s -0.0070 -0.0044 -0.0088  0.0220 -0.0254
ofi_pc1_sum_30s  -0.0074 -0.0018  0.0009  0.0055 -0.0054
ofi_pc1_sum_60s  -0.0036  0.0011 -0.0047  0.0032 -0.0124
ofi_pc2_sum_10s   0.0047  0.0004  0.0048  0.0017 -0.0011
ofi_pc2_sum_300s  0.0040  0.0112 -0.0032  0.0384  0.0093
ofi_pc2_sum_30s   0.0005  0.0008  0.0090  0.0053 -0.0025
ofi_pc2_sum_60s   0.0034  0.0109  0.0146  0.0143  0.0022
horizon           30    60    300   900   1800
feature                                       
ofi_pc1_sum_10s   0.47  0.53  0.67  0.67  0.53
ofi_pc1_sum_300s  0.47  0.60  0.47  0.67  0.47
ofi_pc1_sum_30s   0.47  0.47  0.67  0.60  0.60
ofi_pc1_sum_60s   0.53  0.53  0.53  0.60  0.40
ofi_pc2_sum_10s   0.67  0.67  0.53  0.47  0.40
ofi_pc2_sum_300s  0.60  0.53  0.53  0.53  0.40
ofi_pc2_sum_30s   0.67  0.53  0.60  0.47  0.53
ofi_pc2

In [129]:
direction_col = "ofi_l1_sum_60s_signed_log"
shape_col = "ofi_pc2_sum_60s"

In [130]:
buy_thresh = df_1s[direction_col].quantile(0.80)
sell_thresh = df_1s[direction_col].quantile(0.20)

df_1s["strong_buy_60"] = (
    df_1s[direction_col] >= buy_thresh
)

df_1s["strong_sell_60"] = (
    df_1s[direction_col] <= sell_thresh
)

In [131]:
shape_hi = df_1s[shape_col].quantile(0.70)
shape_lo = df_1s[shape_col].quantile(0.30)

In [132]:
conditions = {
    "buy_shallow": (
        df_1s["strong_buy_60"] &
        (df_1s[shape_col] >= shape_hi)
    ),

    "buy_deep": (
        df_1s["strong_buy_60"] &
        (df_1s[shape_col] <= shape_lo)
    ),

    "sell_shallow": (
        df_1s["strong_sell_60"] &
        (df_1s[shape_col] <= shape_lo)
    ),

    "sell_deep": (
        df_1s["strong_sell_60"] &
        (df_1s[shape_col] >= shape_hi)
    ),
}

In [133]:
df_1s["shape_aligned_60"] = (
    np.sign(df_1s[direction_col])
    *
    df_1s[shape_col]
)

In [134]:
strong_ofi = (
    df_1s["strong_buy_60"] |
    df_1s["strong_sell_60"]
)

hi_shape = df_1s["shape_aligned_60"].quantile(0.70)
lo_shape = df_1s["shape_aligned_60"].quantile(0.30)

In [135]:
for h in [30, 60, 300, 900, 1800]:

    df_1s[f"shape_cont_ret_{h}s"] = (
        np.sign(df_1s[direction_col])
        *
        df_1s[f"fwd_ret_{h}s"]
    )

In [140]:
daily_shape = []

for date, day in df_1s.groupby("date"):

    strong = (
        day["strong_buy_60"] |
        day["strong_sell_60"]
    )

    for h in [30, 60, 300, 900, 1800]:

        target = f"shape_cont_ret_{h}s"

        shallow = day.loc[
            strong &
            (day["shape_aligned_60"] >= hi_shape),
            target
        ].dropna()

        deep = day.loc[
            strong &
            (day["shape_aligned_60"] <= lo_shape),
            target
        ].dropna()

        if len(shallow) < 50 or len(deep) < 50:
            continue

        daily_shape.append({
            "date": date,
            "horizon": h,
            "difference": (
                shallow.mean()
                -
                deep.mean()
            )
        })

daily_shape = pd.DataFrame(daily_shape)

print(
    daily_shape.groupby("horizon").agg(
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=("difference", lambda x: (x > 0).mean()),
        n_days=("difference", "count")
    )
)

         mean_difference  median_difference  positive_days  n_days
horizon                                                           
30             -0.000001          -0.000002       0.266667      15
60              0.000001           0.000002       0.533333      15
300             0.000003           0.000002       0.533333      15
900             0.000005           0.000002       0.533333      15
1800            0.000006          -0.000015       0.466667      15


In [141]:
print(
    daily_shape.groupby("horizon").agg(
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=("difference", lambda x: (x > 0).mean()),
        n_days=("difference", "count")
    )
)

         mean_difference  median_difference  positive_days  n_days
horizon                                                           
30             -0.000001          -0.000002       0.266667      15
60              0.000001           0.000002       0.533333      15
300             0.000003           0.000002       0.533333      15
900             0.000005           0.000002       0.533333      15
1800            0.000006          -0.000015       0.466667      15


In [142]:
g = df_1s.groupby(["symbol", "date"])

df_1s["bid_q1_delta"] = g["bid_q1"].diff()
df_1s["ask_q1_delta"] = g["ask_q1"].diff()

df_1s["bid_depth_l5_delta"] = g["bid_depth_l5"].diff()
df_1s["ask_depth_l5_delta"] = g["ask_depth_l5"].diff()

df_1s["depth_imbalance_delta"] = g["depth_imbalance_l5"].diff()
df_1s["queue_imbalance_delta"] = g["queue_imbalance_l1"].diff()

In [143]:
df_1s["ask_depletion_l1"] = np.maximum(
    -df_1s["ask_q1_delta"], 0
)

df_1s["ask_replenishment_l1"] = np.maximum(
    df_1s["ask_q1_delta"], 0
)

df_1s["bid_depletion_l1"] = np.maximum(
    -df_1s["bid_q1_delta"], 0
)

df_1s["bid_replenishment_l1"] = np.maximum(
    df_1s["bid_q1_delta"], 0
)

In [144]:
prev_bid_q1 = g["bid_q1"].shift(1)
prev_ask_q1 = g["ask_q1"].shift(1)

eps = 1e-12

df_1s["ask_depletion_ratio"] = (
    df_1s["ask_depletion_l1"]
    / (prev_ask_q1 + eps)
)

df_1s["ask_replenishment_ratio"] = (
    df_1s["ask_replenishment_l1"]
    / (prev_ask_q1 + eps)
)

df_1s["bid_depletion_ratio"] = (
    df_1s["bid_depletion_l1"]
    / (prev_bid_q1 + eps)
)

df_1s["bid_replenishment_ratio"] = (
    df_1s["bid_replenishment_l1"]
    / (prev_bid_q1 + eps)
)

In [145]:
for c in [
    "ask_depletion_ratio",
    "ask_replenishment_ratio",
    "bid_depletion_ratio",
    "bid_replenishment_ratio"
]:
    df_1s[f"{c}_log"] = np.log1p(df_1s[c])

In [146]:
df_1s["directional_depletion_l1"] = (
    df_1s["ask_depletion_ratio_log"]
    -
    df_1s["bid_depletion_ratio_log"]
)

In [147]:
df_1s["directional_replenishment_l1"] = (
    df_1s["bid_replenishment_ratio_log"]
    -
    df_1s["ask_replenishment_ratio_log"]
)

In [148]:
for h in [10, 30, 60]:

    df_1s[f"dir_depletion_sum_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["directional_depletion_l1"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=max(1, h // 3)
            ).sum()
        )
    )

    df_1s[f"dir_replenishment_sum_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["directional_replenishment_l1"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=max(1, h // 3)
            ).sum()
        )
    )

In [149]:
for h in [10, 30, 60]:
    df_1s[f"liquidity_pressure_{h}s"] = (
        df_1s[f"dir_depletion_sum_{h}s"]
        +
        df_1s[f"dir_replenishment_sum_{h}s"]
    )

In [150]:
depth_features = [
    "depth_imbalance_l5",
    "queue_imbalance_l1",
    "depth_imbalance_delta",
    "queue_imbalance_delta",
    "dir_depletion_sum_30s",
    "dir_depletion_sum_60s",
    "dir_replenishment_sum_30s",
    "dir_replenishment_sum_60s",
    "liquidity_pressure_30s",
    "liquidity_pressure_60s",
]

In [152]:
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

depth_features = [
    "depth_imbalance_l5",
    "queue_imbalance_l1",
    "depth_imbalance_delta",
    "queue_imbalance_delta",
    "dir_depletion_sum_30s",
    "dir_depletion_sum_60s",
    "dir_replenishment_sum_30s",
    "dir_replenishment_sum_60s",
    "liquidity_pressure_30s",
    "liquidity_pressure_60s",
]

horizons = [30, 60, 300, 900, 1800]

depth_results = []

for date, day in df_1s.groupby("date"):

    for feature in depth_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            depth_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

depth_daily = pd.DataFrame(depth_results)

depth_summary = (
    depth_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [153]:
from scipy.stats import spearmanr
import pandas as pd
import numpy as np

depth_features = [
    "depth_imbalance_l5",
    "queue_imbalance_l1",
    "depth_imbalance_delta",
    "queue_imbalance_delta",
    "dir_depletion_sum_30s",
    "dir_depletion_sum_60s",
    "dir_replenishment_sum_30s",
    "dir_replenishment_sum_60s",
    "liquidity_pressure_30s",
    "liquidity_pressure_60s",
]

horizons = [30, 60, 300, 900, 1800]

depth_results = []

for date, day in df_1s.groupby("date"):

    for feature in depth_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            depth_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

depth_daily = pd.DataFrame(depth_results)

depth_summary = (
    depth_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [154]:
print(
    depth_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    depth_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon                      30      60      300     900     1800
feature                                                          
depth_imbalance_delta      0.0213  0.0192  0.0107  0.0073  0.0052
depth_imbalance_l5         0.0982  0.1074  0.0865  0.0383  0.0498
dir_depletion_sum_30s     -0.0090 -0.0159 -0.0309 -0.0471 -0.0364
dir_depletion_sum_60s     -0.0114 -0.0175 -0.0341 -0.0597 -0.0451
dir_replenishment_sum_30s  0.0878  0.0903  0.0843  0.0710  0.0598
dir_replenishment_sum_60s  0.0808  0.0848  0.0914  0.0780  0.0637
liquidity_pressure_30s     0.0914  0.0905  0.0824  0.0603  0.0511
liquidity_pressure_60s     0.0873  0.0881  0.0931  0.0675  0.0579
queue_imbalance_delta      0.0334  0.0290  0.0178  0.0120  0.0117
queue_imbalance_l1         0.1410  0.1408  0.1016  0.0791  0.0824
horizon                    30    60    300   900   1800
feature                                                
depth_imbalance_delta      1.00   1.0  0.93  1.00  0.80
depth_imbalance_l5         1.00   1.0  0

In [155]:
g = df_1s.groupby(["symbol", "date"])

prev_bid_p1 = g["bid_p1"].shift(1)
prev_ask_p1 = g["ask_p1"].shift(1)

prev_bid_q1 = g["bid_q1"].shift(1)
prev_ask_q1 = g["ask_q1"].shift(1)

same_bid_price = df_1s["bid_p1"] == prev_bid_p1
same_ask_price = df_1s["ask_p1"] == prev_ask_p1

bid_q_change_same = np.where(
    same_bid_price,
    df_1s["bid_q1"] - prev_bid_q1,
    0.0
)

ask_q_change_same = np.where(
    same_ask_price,
    df_1s["ask_q1"] - prev_ask_q1,
    0.0
)

In [156]:
df_1s["bid_replenish_same"] = np.maximum(
    bid_q_change_same, 0
)

df_1s["ask_replenish_same"] = np.maximum(
    ask_q_change_same, 0
)

In [157]:
df_1s["bid_deplete_same"] = np.maximum(
    -bid_q_change_same, 0
)

df_1s["ask_deplete_same"] = np.maximum(
    -ask_q_change_same, 0
)

In [158]:
eps = 1e-12

df_1s["bid_replenish_same_ratio"] = (
    df_1s["bid_replenish_same"]
    / (prev_bid_q1 + eps)
)

df_1s["ask_replenish_same_ratio"] = (
    df_1s["ask_replenish_same"]
    / (prev_ask_q1 + eps)
)

df_1s["bid_deplete_same_ratio"] = (
    df_1s["bid_deplete_same"]
    / (prev_bid_q1 + eps)
)

df_1s["ask_deplete_same_ratio"] = (
    df_1s["ask_deplete_same"]
    / (prev_ask_q1 + eps)
)

In [159]:
for c in [
    "bid_replenish_same_ratio",
    "ask_replenish_same_ratio",
    "bid_deplete_same_ratio",
    "ask_deplete_same_ratio"
]:
    df_1s[f"{c}_log"] = np.log1p(df_1s[c])

In [160]:
df_1s["bid_price_up"] = (
    df_1s["bid_p1"] > prev_bid_p1
).astype(int)

df_1s["bid_price_down"] = (
    df_1s["bid_p1"] < prev_bid_p1
).astype(int)

In [161]:
df_1s["ask_price_down"] = (
    df_1s["ask_p1"] < prev_ask_p1
).astype(int)

df_1s["ask_price_up"] = (
    df_1s["ask_p1"] > prev_ask_p1
).astype(int)

In [162]:
df_1s["queue_replenishment_pressure"] = (
    df_1s["bid_replenish_same_ratio_log"]
    -
    df_1s["ask_replenish_same_ratio_log"]
)

In [163]:
df_1s["queue_depletion_pressure"] = (
    df_1s["ask_deplete_same_ratio_log"]
    -
    df_1s["bid_deplete_same_ratio_log"]
)

In [164]:
for h in [30, 60]:

    df_1s[f"queue_replenishment_pressure_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["queue_replenishment_pressure"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=h//3
            ).sum()
        )
    )

    df_1s[f"queue_depletion_pressure_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["queue_depletion_pressure"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=h//3
            ).sum()
        )
    )

In [165]:
df_1s["quote_move_pressure"] = (
    df_1s["bid_price_up"]
    + df_1s["ask_price_down"]
    - df_1s["bid_price_down"]
    - df_1s["ask_price_up"]
)

In [166]:
for h in [30, 60]:

    df_1s[f"quote_move_pressure_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["quote_move_pressure"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=h//3
            ).sum()
        )
    )

In [167]:
queue_features_v2 = [
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "queue_replenishment_pressure_30s",
    "queue_replenishment_pressure_60s",
    "queue_depletion_pressure_30s",
    "queue_depletion_pressure_60s",
    "quote_move_pressure_30s",
    "quote_move_pressure_60s",
]

In [168]:
queue_v2_results = []

for date, day in df_1s.groupby("date"):

    for feature in queue_features_v2:

        for h in [30, 60, 300, 900, 1800]:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            queue_v2_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

queue_v2_daily = pd.DataFrame(queue_v2_results)

queue_v2_summary = (
    queue_v2_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [169]:
print(
    queue_v2_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    queue_v2_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon                             30      60      300     900     1800
feature                                                                 
depth_imbalance_delta             0.0213  0.0192  0.0107  0.0073  0.0052
depth_imbalance_l5                0.0982  0.1074  0.0865  0.0383  0.0498
queue_depletion_pressure_30s     -0.0108 -0.0068 -0.0158 -0.0394 -0.0319
queue_depletion_pressure_60s     -0.0032 -0.0010 -0.0176 -0.0489 -0.0391
queue_imbalance_delta             0.0334  0.0290  0.0178  0.0120  0.0117
queue_imbalance_l1                0.1410  0.1408  0.1016  0.0791  0.0824
queue_replenishment_pressure_30s  0.0703  0.0706  0.0593  0.0697  0.0519
queue_replenishment_pressure_60s  0.0593  0.0639  0.0614  0.0789  0.0565
quote_move_pressure_30s           0.0138  0.0208  0.0136  0.0186  0.0047
quote_move_pressure_60s           0.0207  0.0275  0.0173  0.0249  0.0070
horizon                           30    60    300   900   1800
feature                                                      

In [170]:
ofi60 = "ofi_l1_sum_60s_signed_log"

df_1s["ofi_aligned_qi"] = (
    np.sign(df_1s[ofi60])
    *
    df_1s["queue_imbalance_l1"]
)

In [171]:
df_1s["ofi_aligned_di"] = (
    np.sign(df_1s[ofi60])
    *
    df_1s["depth_imbalance_l5"]
)

In [172]:
df_1s["ofi_aligned_replenishment"] = (
    np.sign(df_1s[ofi60])
    *
    df_1s["queue_replenishment_pressure_60s"]
)

In [173]:
buy_thr = df_1s[ofi60].quantile(0.80)
sell_thr = df_1s[ofi60].quantile(0.20)

strong_ofi = (
    (df_1s[ofi60] >= buy_thr)
    |
    (df_1s[ofi60] <= sell_thr)
)

In [174]:
interaction_features = [
    "ofi_aligned_qi",
    "ofi_aligned_di",
    "ofi_aligned_replenishment"
]

horizons = [30, 60, 300, 900, 1800]

interaction_rows = []

for feature in interaction_features:

    hi = df_1s[feature].quantile(0.70)
    lo = df_1s[feature].quantile(0.30)

    for h in horizons:

        cont = (
            np.sign(df_1s[ofi60])
            *
            df_1s[f"fwd_ret_{h}s"]
        )

        agree = cont[
            strong_ofi &
            (df_1s[feature] >= hi)
        ].dropna()

        disagree = cont[
            strong_ofi &
            (df_1s[feature] <= lo)
        ].dropna()

        interaction_rows.append({
            "feature": feature,
            "horizon": h,
            "agree_mean": agree.mean(),
            "disagree_mean": disagree.mean(),
            "difference": agree.mean() - disagree.mean(),
            "agree_cont_frac": (agree > 0).mean(),
            "disagree_cont_frac": (disagree > 0).mean(),
            "n_agree": len(agree),
            "n_disagree": len(disagree)
        })

interaction_results = pd.DataFrame(interaction_rows)

print(
    interaction_results.pivot(
        index="feature",
        columns="horizon",
        values="difference"
    )
)

print(
    interaction_results.pivot(
        index="feature",
        columns="horizon",
        values="agree_cont_frac"
    )
)

print(
    interaction_results.pivot(
        index="feature",
        columns="horizon",
        values="disagree_cont_frac"
    )
)

horizon                        30        60        300       900       1800
feature                                                                    
ofi_aligned_di             0.000014  0.000020  0.000046  0.000082  0.000155
ofi_aligned_qi             0.000018  0.000024  0.000040  0.000071  0.000102
ofi_aligned_replenishment  0.000012  0.000018  0.000031  0.000056  0.000046
horizon                        30        60        300       900       1800
feature                                                                    
ofi_aligned_di             0.392357  0.460035  0.538685  0.559097  0.561270
ofi_aligned_qi             0.400341  0.467246  0.525118  0.542904  0.548411
ofi_aligned_replenishment  0.378274  0.436082  0.508526  0.531844  0.512561
horizon                        30        60        300       900       1800
feature                                                                    
ofi_aligned_di             0.294219  0.344128  0.402638  0.416601  0.383928
ofi_aligned_

In [175]:
daily_interactions = []

interaction_features = [
    "ofi_aligned_qi",
    "ofi_aligned_di",
    "ofi_aligned_replenishment"
]

for feature in interaction_features:

    hi = df_1s[feature].quantile(0.70)
    lo = df_1s[feature].quantile(0.30)

    for date, day in df_1s.groupby("date"):

        strong_day = (
            (day[ofi60] >= buy_thr)
            |
            (day[ofi60] <= sell_thr)
        )

        for h in [30, 60, 300, 900, 1800]:

            continuation = (
                np.sign(day[ofi60])
                *
                day[f"fwd_ret_{h}s"]
            )

            agree = continuation[
                strong_day &
                (day[feature] >= hi)
            ].dropna()

            disagree = continuation[
                strong_day &
                (day[feature] <= lo)
            ].dropna()

            if len(agree) < 30 or len(disagree) < 30:
                continue

            daily_interactions.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "agree_mean": agree.mean(),
                "disagree_mean": disagree.mean(),
                "difference": agree.mean() - disagree.mean(),
                "n_agree": len(agree),
                "n_disagree": len(disagree)
            })

daily_interactions = pd.DataFrame(daily_interactions)

In [176]:
daily_interaction_summary = (
    daily_interactions
    .groupby(["feature", "horizon"])
    .agg(
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        std_difference=("difference", "std"),
        positive_days=("difference", lambda x: (x > 0).mean()),
        n_days=("difference", "count")
    )
    .reset_index()
)

print(
    daily_interaction_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_difference"
    )
)

print(
    daily_interaction_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon                        30        60        300       900       1800
feature                                                                    
ofi_aligned_di             0.000013  0.000018  0.000042  0.000069  0.000134
ofi_aligned_qi             0.000018  0.000023  0.000037  0.000064  0.000092
ofi_aligned_replenishment  0.000011  0.000016  0.000028  0.000055  0.000049
horizon                    30    60    300   900   1800
feature                                                
ofi_aligned_di             1.00  1.00  0.93  0.80  0.80
ofi_aligned_qi             1.00  1.00  0.93  0.93  0.93
ofi_aligned_replenishment  0.93  0.93  0.87  0.87  0.73


In [177]:
df_1s["log_spread_bps"] = np.log1p(
    df_1s["spread_bps_calc"]
)

In [178]:
g = df_1s.groupby(["symbol", "date"])

df_1s["spread_change_1s"] = (
    g["log_spread_bps"].diff()
)

In [179]:
for h in [30, 60, 300]:

    df_1s[f"spread_change_{h}s"] = (
        df_1s["log_spread_bps"]
        -
        g["log_spread_bps"].shift(h)
    )

In [180]:
def rolling_percentile_rank(s, window=1800, min_periods=300):
    return s.rolling(
        window,
        min_periods=min_periods
    ).rank(pct=True)

df_1s["spread_rank_1800s"] = (
    df_1s
    .groupby(["symbol", "date"])["spread_bps_calc"]
    .transform(
        lambda x: rolling_percentile_rank(
            x,
            window=1800,
            min_periods=300
        )
    )
)

In [181]:
df_1s["spread_expansion_30s"] = (
    df_1s["spread_change_30s"]
)

df_1s["spread_expansion_60s"] = (
    df_1s["spread_change_60s"]
)

In [182]:
spread_features = [
    "log_spread_bps",
    "spread_rank_1800s",
    "spread_change_1s",
    "spread_expansion_30s",
    "spread_expansion_60s",
]

In [183]:
spread_results = []

for date, day in df_1s.groupby("date"):

    for feature in spread_features:

        for h in [30, 60, 300, 900, 1800]:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            spread_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

spread_daily = pd.DataFrame(spread_results)

spread_summary = (
    spread_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [184]:
print(
    spread_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    spread_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon                 30      60      300     900     1800
feature                                                     
log_spread_bps        0.0105  0.0207  0.0372  0.0457  0.0588
spread_change_1s     -0.0065 -0.0042 -0.0029 -0.0036 -0.0008
spread_expansion_30s -0.0014 -0.0087  0.0067 -0.0054 -0.0021
spread_expansion_60s -0.0064 -0.0037  0.0158 -0.0017 -0.0008
spread_rank_1800s     0.0112  0.0182  0.0159 -0.0014 -0.0019
horizon               30    60    300   900   1800
feature                                           
log_spread_bps        0.67  0.80  0.80  0.60   0.8
spread_change_1s      0.40  0.33  0.27  0.40   0.4
spread_expansion_30s  0.33  0.27  0.47  0.33   0.4
spread_expansion_60s  0.33  0.60  0.67  0.47   0.6
spread_rank_1800s     0.67  0.67  0.60  0.33   0.4


In [185]:
ofi60 = "ofi_l1_sum_60s_signed_log"

buy_thr = df_1s[ofi60].quantile(0.80)
sell_thr = df_1s[ofi60].quantile(0.20)

strong_ofi = (
    (df_1s[ofi60] >= buy_thr)
    |
    (df_1s[ofi60] <= sell_thr)
)

In [186]:
tight = df_1s["spread_rank_1800s"] <= 0.30
wide = df_1s["spread_rank_1800s"] >= 0.70

In [187]:
for h in [30, 60, 300, 900, 1800]:
    df_1s[f"spread_cont_ret_{h}s"] = (
        np.sign(df_1s[ofi60])
        *
        df_1s[f"fwd_ret_{h}s"]
    )

In [188]:
spread_interaction = []

for h in [30, 60, 300, 900, 1800]:

    target = f"spread_cont_ret_{h}s"

    tight_vals = df_1s.loc[
        strong_ofi & tight,
        target
    ].dropna()

    wide_vals = df_1s.loc[
        strong_ofi & wide,
        target
    ].dropna()

    spread_interaction.append({
        "horizon": h,
        "tight_mean": tight_vals.mean(),
        "wide_mean": wide_vals.mean(),
        "difference_tight_minus_wide":
            tight_vals.mean() - wide_vals.mean(),
        "tight_cont_frac": (tight_vals > 0).mean(),
        "wide_cont_frac": (wide_vals > 0).mean(),
        "n_tight": len(tight_vals),
        "n_wide": len(wide_vals)
    })

spread_interaction = pd.DataFrame(spread_interaction)

print(spread_interaction)

   horizon  tight_mean  wide_mean  difference_tight_minus_wide  \
0       30    0.000004   0.000004                 4.693284e-07   
1       60    0.000007   0.000004                 2.555506e-06   
2      300    0.000015   0.000004                 1.124137e-05   
3      900    0.000031  -0.000001                 3.207648e-05   
4     1800    0.000040  -0.000010                 5.010894e-05   

   tight_cont_frac  wide_cont_frac  n_tight  n_wide  
0         0.350232        0.354221    53987   49311  
1         0.408894        0.408923    53926   49266  
2         0.486618        0.482526    53693   49187  
3         0.517857        0.471062    53115   48794  
4         0.520428        0.466497    52013   48071  


In [189]:
daily_spread_interaction = []

for date, day in df_1s.groupby("date"):

    strong_day = (
        (day[ofi60] >= buy_thr)
        |
        (day[ofi60] <= sell_thr)
    )

    tight_day = day["spread_rank_1800s"] <= 0.30
    wide_day = day["spread_rank_1800s"] >= 0.70

    for h in [30, 60, 300, 900, 1800]:

        cont = (
            np.sign(day[ofi60])
            *
            day[f"fwd_ret_{h}s"]
        )

        a = cont[strong_day & tight_day].dropna()
        b = cont[strong_day & wide_day].dropna()

        if len(a) < 30 or len(b) < 30:
            continue

        daily_spread_interaction.append({
            "date": date,
            "horizon": h,
            "difference": a.mean() - b.mean()
        })

daily_spread_interaction = pd.DataFrame(
    daily_spread_interaction
)

print(
    daily_spread_interaction
    .groupby("horizon")
    .agg(
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=("difference", lambda x: (x > 0).mean()),
        n_days=("difference", "count")
    )
)

         mean_difference  median_difference  positive_days  n_days
horizon                                                           
30          7.146533e-07           0.000001       0.600000      15
60          3.152331e-06           0.000004       0.666667      15
300         1.042386e-05           0.000003       0.666667      15
900         3.104603e-05           0.000022       0.866667      15
1800        4.570431e-05           0.000057       0.866667      15


In [190]:
print(
    df_1s["volume_delta"].describe(
        percentiles=[
            .001, .01, .05,
            .25, .5, .75,
            .95, .99, .999
        ]
    )
)

print(
    "Zero %:",
    100 * (df_1s["volume_delta"] == 0).mean()
)

count    409201.000000
mean          4.253560
std         110.478034
min           0.000000
0.1%          0.000000
1%            0.000000
5%            0.000000
25%           0.000000
50%           0.000000
75%           0.000000
95%           0.000000
99%          13.000000
99.9%      1000.000000
max       12408.000000
Name: volume_delta, dtype: float64
Zero %: 98.58187052328806


In [191]:
df_1s["volume_delta_sl"] = (
    np.sign(df_1s["volume_delta"])
    *
    np.log1p(np.abs(df_1s["volume_delta"]))
)

In [192]:
for h in [10, 30, 60, 300]:

    df_1s[f"volume_delta_sum_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["volume_delta"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=max(1, h // 3)
            ).sum()
        )
    )

    c = f"volume_delta_sum_{h}s"

    df_1s[f"{c}_sl"] = (
        np.sign(df_1s[c])
        *
        np.log1p(np.abs(df_1s[c]))
    )

In [193]:
df_1s["volume_delta_rate_30s"] = (
    df_1s["volume_delta_sum_30s"] / 30
)

df_1s["volume_delta_rate_300s"] = (
    df_1s["volume_delta_sum_300s"] / 300
)

df_1s["volume_delta_rate_diff_30_300"] = (
    df_1s["volume_delta_rate_30s"]
    -
    df_1s["volume_delta_rate_300s"]
)

df_1s["volume_delta_rate_diff_30_300_sl"] = (
    np.sign(df_1s["volume_delta_rate_diff_30_300"])
    *
    np.log1p(
        np.abs(df_1s["volume_delta_rate_diff_30_300"])
    )
)

In [194]:
for h in [30, 60, 300]:

    df_1s[f"tick_count_sum_{h}s"] = (
        df_1s
        .groupby(["symbol", "date"])["tick_count"]
        .transform(
            lambda x: x.rolling(
                h,
                min_periods=max(1, h // 3)
            ).sum()
        )
    )

In [195]:
for h in [30, 60, 300]:
    df_1s[f"log_tick_count_{h}s"] = np.log1p(
        df_1s[f"tick_count_sum_{h}s"]
    )

In [196]:
eps = 1e-12

for h in [30, 60]:

    df_1s[f"signed_volume_per_tick_{h}s"] = (
        df_1s[f"volume_delta_sum_{h}s"]
        /
        (
            df_1s[f"tick_count_sum_{h}s"]
            + eps
        )
    )

    c = f"signed_volume_per_tick_{h}s"

    df_1s[f"{c}_sl"] = (
        np.sign(df_1s[c])
        *
        np.log1p(np.abs(df_1s[c]))
    )

In [197]:
df_1s["ofi_volume_sign_agreement"] = (
    np.sign(df_1s["ofi_l1_sum_60s_signed_log"])
    *
    np.sign(df_1s["volume_delta_sum_60s"])
)

In [198]:
df_1s["ofi_aligned_volume_delta_60s"] = (
    np.sign(df_1s["ofi_l1_sum_60s_signed_log"])
    *
    df_1s["volume_delta_sum_60s_sl"]
)

In [199]:
volume_features = [
    "volume_delta_sl",
    "volume_delta_sum_10s_sl",
    "volume_delta_sum_30s_sl",
    "volume_delta_sum_60s_sl",
    "volume_delta_sum_300s_sl",
    "volume_delta_rate_diff_30_300_sl",
    "signed_volume_per_tick_30s_sl",
    "signed_volume_per_tick_60s_sl",
    "log_tick_count_30s",
    "log_tick_count_60s",
    "log_tick_count_300s",
]

In [201]:
volume_results = []

for date, day in df_1s.groupby("date"):

    for feature in volume_features:

        for h in [30, 60, 300, 900, 1800]:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            volume_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

volume_daily = pd.DataFrame(volume_results)

volume_summary = (
    volume_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [202]:
print(
    volume_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    volume_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon                             30      60      300     900     1800
feature                                                                 
log_tick_count_300s              -0.0042 -0.0096 -0.0069 -0.0201 -0.0321
log_tick_count_30s               -0.0158 -0.0242 -0.0192 -0.0354 -0.0205
log_tick_count_60s               -0.0144 -0.0208 -0.0167 -0.0328 -0.0214
signed_volume_per_tick_30s_sl    -0.0114 -0.0213 -0.0129 -0.0034 -0.0008
signed_volume_per_tick_60s_sl    -0.0142 -0.0222 -0.0048  0.0024  0.0019
volume_delta_rate_diff_30_300_sl -0.0081 -0.0106 -0.0284 -0.0141 -0.0042
volume_delta_sl                  -0.0089 -0.0076 -0.0075 -0.0047 -0.0057
volume_delta_sum_10s_sl          -0.0087 -0.0115 -0.0128 -0.0074 -0.0048
volume_delta_sum_300s_sl         -0.0012 -0.0035  0.0175  0.0142  0.0028
volume_delta_sum_30s_sl          -0.0115 -0.0215 -0.0134 -0.0038 -0.0009
volume_delta_sum_60s_sl          -0.0147 -0.0230 -0.0061  0.0017  0.0020
horizon                           30    60    300  

In [203]:
ofi60 = "ofi_l1_sum_60s_signed_log"

buy_thr = df_1s[ofi60].quantile(0.80)
sell_thr = df_1s[ofi60].quantile(0.20)

strong_ofi = (
    (df_1s[ofi60] >= buy_thr)
    |
    (df_1s[ofi60] <= sell_thr)
)

In [204]:
flow_col = "ofi_aligned_volume_delta_60s"

hi = df_1s[flow_col].quantile(0.70)
lo = df_1s[flow_col].quantile(0.30)

In [205]:
for h in [30, 60, 300, 900, 1800]:

    df_1s[f"flow_cont_ret_{h}s"] = (
        np.sign(df_1s[ofi60])
        *
        df_1s[f"fwd_ret_{h}s"]
    )

In [206]:
rows = []

for h in [30, 60, 300, 900, 1800]:

    target = f"flow_cont_ret_{h}s"

    confirm = df_1s.loc[
        strong_ofi &
        (df_1s[flow_col] >= hi),
        target
    ].dropna()

    oppose = df_1s.loc[
        strong_ofi &
        (df_1s[flow_col] <= lo),
        target
    ].dropna()

    rows.append({
        "horizon": h,
        "confirm_mean": confirm.mean(),
        "oppose_mean": oppose.mean(),
        "difference": confirm.mean() - oppose.mean(),
        "confirm_cont_frac": (confirm > 0).mean(),
        "oppose_cont_frac": (oppose > 0).mean(),
        "n_confirm": len(confirm),
        "n_oppose": len(oppose)
    })

flow_interaction = pd.DataFrame(rows)

print(flow_interaction)

   horizon  confirm_mean  oppose_mean    difference  confirm_cont_frac  \
0       30      0.000004     0.000005 -1.149181e-06           0.341107   
1       60      0.000005     0.000007 -1.402479e-06           0.403595   
2      300      0.000007     0.000006  9.548456e-07           0.487993   
3      900      0.000017     0.000008  8.583027e-06           0.508145   
4     1800      0.000017     0.000001  1.617337e-05           0.503869   

   oppose_cont_frac  n_confirm  n_oppose  
0          0.339590     131648    133184  
1          0.399124     131560    133109  
2          0.470922     131137    132625  
3          0.477854     129896    131289  
4          0.469719     128053    129075  


In [207]:
daily_flow_interaction = []

for date, day in df_1s.groupby("date"):

    strong_day = (
        (day[ofi60] >= buy_thr)
        |
        (day[ofi60] <= sell_thr)
    )

    for h in [30, 60, 300, 900, 1800]:

        cont = (
            np.sign(day[ofi60])
            *
            day[f"fwd_ret_{h}s"]
        )

        confirm = cont[
            strong_day &
            (day[flow_col] >= hi)
        ].dropna()

        oppose = cont[
            strong_day &
            (day[flow_col] <= lo)
        ].dropna()

        if len(confirm) < 30 or len(oppose) < 30:
            continue

        daily_flow_interaction.append({
            "date": date,
            "horizon": h,
            "difference": (
                confirm.mean() - oppose.mean()
            )
        })

daily_flow_interaction = pd.DataFrame(
    daily_flow_interaction
)

print(
    daily_flow_interaction
    .groupby("horizon")
    .agg(
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=("difference", lambda x: (x > 0).mean()),
        n_days=("difference", "count")
    )
)

         mean_difference  median_difference  positive_days  n_days
horizon                                                           
30         -1.257057e-06      -1.228284e-06       0.400000      15
60         -1.633173e-06      -4.013983e-08       0.466667      15
300        -3.020267e-07      -2.309251e-06       0.333333      15
900         5.794442e-06       5.838457e-06       0.600000      15
1800        1.230234e-05       1.439202e-05       0.666667      15


In [208]:
df_1s["vwap_gap"] = (
    (df_1s["mid"] - df_1s["vwap"])
    / df_1s["mid"]
)

In [209]:
eps = 1e-12

df_1s["vol_ratio_30_300"] = (
    df_1s["realized_vol_30s"]
    / (df_1s["realized_vol_300s"] + eps)
)

df_1s["vol_ratio_60_300"] = (
    df_1s["realized_vol_60s"]
    / (df_1s["realized_vol_300s"] + eps)
)

In [210]:
for h in [10, 30, 60, 300]:
    df_1s[f"log_realized_vol_{h}s"] = np.log1p(
        df_1s[f"realized_vol_{h}s"]
    )

In [211]:
vol_features = [
    "vwap_gap",
    "log_realized_vol_10s",
    "log_realized_vol_30s",
    "log_realized_vol_60s",
    "log_realized_vol_300s",
    "vol_ratio_30_300",
    "vol_ratio_60_300",
    "price_mom_10s",
    "price_mom_30s",
    "price_mom_60s",
]

In [212]:
horizons = [30, 60, 300, 900, 1800]

vol_results = []

for date, day in df_1s.groupby("date"):

    for feature in vol_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            vol_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

vol_daily = pd.DataFrame(vol_results)

In [213]:
vol_summary = (
    vol_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [214]:
print(
    vol_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    vol_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon                  30      60      300     900     1800
feature                                                      
log_realized_vol_10s  -0.0097 -0.0044 -0.0097  0.0046  0.0104
log_realized_vol_300s  0.0173  0.0251  0.0632  0.0604  0.0447
log_realized_vol_30s  -0.0078 -0.0116 -0.0013  0.0128  0.0184
log_realized_vol_60s  -0.0081 -0.0012  0.0168  0.0263  0.0315
price_mom_10s         -0.0259 -0.0145 -0.0140 -0.0043 -0.0142
price_mom_30s         -0.0140 -0.0108 -0.0089  0.0013 -0.0130
price_mom_60s         -0.0220 -0.0243 -0.0159 -0.0014 -0.0213
vol_ratio_30_300      -0.0134 -0.0166 -0.0112  0.0048  0.0193
vol_ratio_60_300      -0.0185 -0.0117 -0.0031  0.0045  0.0255
vwap_gap              -0.0876 -0.1069 -0.1978 -0.2660 -0.3817
horizon                30    60    300   900   1800
feature                                            
log_realized_vol_10s   0.33  0.47  0.47  0.60  0.73
log_realized_vol_300s  0.73  0.67  0.67  0.73  0.60
log_realized_vol_30s   0.33  0.40  0.60  0.60  0

In [215]:
g = df_1s.groupby(["symbol", "date"])

for h in [10, 30, 60]:
    df_1s[f"price_mom_{h}s_new"] = (
        df_1s["log_mid"]
        - g["log_mid"].shift(h)
    )

In [216]:
df_1s["_ret_sq"] = df_1s["ret_1s"] ** 2

for h in [10, 30, 60, 300]:
    df_1s[f"rv_{h}s_new"] = (
        df_1s
        .groupby(["symbol", "date"])["_ret_sq"]
        .transform(
            lambda x: np.sqrt(
                x.rolling(
                    h,
                    min_periods=max(2, h // 3)
                ).sum()
            )
        )
    )

In [217]:
for h in [10, 30, 60, 300]:
    df_1s[f"log_rv_{h}s_new"] = np.log1p(
        df_1s[f"rv_{h}s_new"]
    )

In [218]:
eps = 1e-12

df_1s["rv_ratio_30_300_new"] = (
    df_1s["rv_30s_new"]
    / (df_1s["rv_300s_new"] + eps)
)

df_1s["rv_ratio_60_300_new"] = (
    df_1s["rv_60s_new"]
    / (df_1s["rv_300s_new"] + eps)
)

In [219]:
vol_features_v2 = [
    "log_rv_10s_new",
    "log_rv_30s_new",
    "log_rv_60s_new",
    "log_rv_300s_new",
    "rv_ratio_30_300_new",
    "rv_ratio_60_300_new",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
]

In [220]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

eps = 1e-12

# ==========================================
# 1. REBUILD MOMENTUM ON THE TRUE 1-SEC GRID
# ==========================================

g = df_1s.groupby(["symbol", "date"])

for h in [10, 30, 60]:
    df_1s[f"price_mom_{h}s_new"] = (
        df_1s["log_mid"]
        - g["log_mid"].shift(h)
    )

# ==========================================
# 2. REBUILD REALIZED VOLATILITY
# ==========================================

df_1s["_ret_sq"] = df_1s["ret_1s"] ** 2

for h in [10, 30, 60, 300]:

    df_1s[f"rv_{h}s_new"] = (
        df_1s
        .groupby(["symbol", "date"])["_ret_sq"]
        .transform(
            lambda x: np.sqrt(
                x.rolling(
                    h,
                    min_periods=max(2, h // 3)
                ).sum()
            )
        )
    )

# ==========================================
# 3. LOG VOLATILITY
# ==========================================

for h in [10, 30, 60, 300]:

    df_1s[f"log_rv_{h}s_new"] = np.log1p(
        df_1s[f"rv_{h}s_new"]
    )

# ==========================================
# 4. VOLATILITY TERM-STRUCTURE FEATURES
# ==========================================

df_1s["rv_ratio_30_300_new"] = (
    df_1s["rv_30s_new"]
    / (df_1s["rv_300s_new"] + eps)
)

df_1s["rv_ratio_60_300_new"] = (
    df_1s["rv_60s_new"]
    / (df_1s["rv_300s_new"] + eps)
)

# ==========================================
# 5. FEATURES TO TEST
# ==========================================

vol_features_v2 = [
    "log_rv_10s_new",
    "log_rv_30s_new",
    "log_rv_60s_new",
    "log_rv_300s_new",

    "rv_ratio_30_300_new",
    "rv_ratio_60_300_new",

    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
]

# ==========================================
# 6. DAILY SPEARMAN IC
# ==========================================

horizons = [30, 60, 300, 900, 1800]

vol_v2_results = []

for date, day in df_1s.groupby("date"):

    for feature in vol_features_v2:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            vol_v2_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

vol_v2_daily = pd.DataFrame(vol_v2_results)

# ==========================================
# 7. SUMMARY
# ==========================================

vol_v2_summary = (
    vol_v2_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

# ==========================================
# 8. PRINT TABLES
# ==========================================

print(
    vol_v2_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    vol_v2_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon                30      60      300     900     1800
feature                                                    
log_rv_10s_new      -0.0071 -0.0098 -0.0081 -0.0034  0.0023
log_rv_300s_new      0.0080  0.0102  0.0148  0.0150  0.0265
log_rv_30s_new      -0.0032 -0.0075 -0.0022  0.0003  0.0040
log_rv_60s_new      -0.0060 -0.0069  0.0011  0.0019  0.0038
price_mom_10s_new   -0.0838 -0.0780 -0.0402 -0.0213 -0.0230
price_mom_30s_new   -0.1065 -0.0981 -0.0573 -0.0320 -0.0325
price_mom_60s_new   -0.0978 -0.0938 -0.0659 -0.0351 -0.0393
rv_ratio_30_300_new -0.0108 -0.0156 -0.0086 -0.0078 -0.0011
rv_ratio_60_300_new -0.0140 -0.0154 -0.0088 -0.0106 -0.0045
horizon              30    60    300   900   1800
feature                                          
log_rv_10s_new       0.40  0.40  0.33  0.47  0.60
log_rv_300s_new      0.73  0.53  0.67  0.53  0.60
log_rv_30s_new       0.47  0.47  0.47  0.60  0.60
log_rv_60s_new       0.40  0.40  0.53  0.60  0.53
price_mom_10s_new    0.00  0.00  0.07  0

In [221]:
df_1s["mean_reversion_signal_30s"] = (
    -df_1s["price_mom_30s_new"]
)

In [222]:
ofi60 = "ofi_l1_sum_60s_signed_log"

df_1s["ofi_aligned_mean_reversion"] = (
    np.sign(df_1s[ofi60])
    *
    df_1s["mean_reversion_signal_30s"]
)

In [223]:
buy_thr = df_1s[ofi60].quantile(0.80)
sell_thr = df_1s[ofi60].quantile(0.20)

strong_ofi = (
    (df_1s[ofi60] >= buy_thr)
    |
    (df_1s[ofi60] <= sell_thr)
)

In [224]:
mr_col = "ofi_aligned_mean_reversion"

hi = df_1s[mr_col].quantile(0.70)
lo = df_1s[mr_col].quantile(0.30)

In [225]:
rows = []

for h in [30, 60, 300, 900, 1800]:

    cont = (
        np.sign(df_1s[ofi60])
        *
        df_1s[f"fwd_ret_{h}s"]
    )

    agree = cont[
        strong_ofi &
        (df_1s[mr_col] >= hi)
    ].dropna()

    conflict = cont[
        strong_ofi &
        (df_1s[mr_col] <= lo)
    ].dropna()

    rows.append({
        "horizon": h,
        "agree_mean": agree.mean(),
        "conflict_mean": conflict.mean(),
        "difference": agree.mean() - conflict.mean(),
        "agree_cont_frac": (agree > 0).mean(),
        "conflict_cont_frac": (conflict > 0).mean(),
        "n_agree": len(agree),
        "n_conflict": len(conflict)
    })

mr_interaction = pd.DataFrame(rows)

print(mr_interaction)

   horizon  agree_mean  conflict_mean  difference  agree_cont_frac  \
0       30    0.000007       0.000001    0.000006         0.350583   
1       60    0.000010       0.000002    0.000007         0.422978   
2      300    0.000012       0.000007    0.000005         0.499077   
3      900    0.000011       0.000019   -0.000008         0.500803   
4     1800    0.000011       0.000013   -0.000003         0.490794   

   conflict_cont_frac  n_agree  n_conflict  
0            0.336785    67365       73774  
1            0.381802    67344       73745  
2            0.460661    67186       73503  
3            0.487681    66657       72816  
4            0.483089    65504       71730  


In [226]:
daily_mr = []

for date, day in df_1s.groupby("date"):

    strong_day = (
        (day[ofi60] >= buy_thr)
        |
        (day[ofi60] <= sell_thr)
    )

    for h in [30, 60, 300, 900, 1800]:

        cont = (
            np.sign(day[ofi60])
            *
            day[f"fwd_ret_{h}s"]
        )

        agree = cont[
            strong_day &
            (day[mr_col] >= hi)
        ].dropna()

        conflict = cont[
            strong_day &
            (day[mr_col] <= lo)
        ].dropna()

        if len(agree) < 30 or len(conflict) < 30:
            continue

        daily_mr.append({
            "date": date,
            "horizon": h,
            "difference": (
                agree.mean() - conflict.mean()
            )
        })

daily_mr = pd.DataFrame(daily_mr)

print(
    daily_mr.groupby("horizon").agg(
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=("difference", lambda x: (x > 0).mean()),
        n_days=("difference", "count")
    )
)

         mean_difference  median_difference  positive_days  n_days
horizon                                                           
30              0.000008           0.000008       0.800000      15
60              0.000011           0.000012       0.866667      15
300             0.000008           0.000004       0.666667      15
900            -0.000004          -0.000001       0.466667      15
1800            0.000003           0.000004       0.600000      15


In [228]:
window = 300
min_periods = 120

for lag in [1, 5, 10]:

    lagged_col = f"_ret_lag_{lag}"

    df_1s[lagged_col] = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .shift(lag)
    )

    df_1s[f"ret_acf_lag{lag}_300s"] = (
        df_1s
        .groupby(["symbol", "date"], group_keys=False)
        .apply(
            lambda g: g["ret_1s"]
            .rolling(
                window=window,
                min_periods=min_periods
            )
            .corr(g[lagged_col])
        )
    )

In [229]:
df_1s["ret_acf_slope_1_10"] = (
    df_1s["ret_acf_lag1_300s"]
    -
    df_1s["ret_acf_lag10_300s"]
)

In [230]:
acf_features = [
    "ret_acf_lag1_300s",
    "ret_acf_lag5_300s",
    "ret_acf_lag10_300s",
    "ret_acf_slope_1_10"
]

print(
    df_1s[acf_features].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

       ret_acf_lag1_300s  ret_acf_lag5_300s  ret_acf_lag10_300s  \
count      404358.000000      404222.000000       404053.000000   
mean           -0.031377          -0.022235           -0.009375   
std             0.092256           0.081351            0.063146   
min            -0.986408          -0.785774           -0.833720   
1%             -0.375833          -0.324466           -0.227458   
5%             -0.215044          -0.177978           -0.122365   
25%            -0.052812          -0.038738           -0.021685   
50%            -0.002424          -0.001723           -0.001112   
75%             0.000000           0.000190            0.006688   
95%             0.081341           0.078651            0.083392   
99%             0.169582           0.163649            0.146258   
max             0.705909           0.799291            0.593347   

       ret_acf_slope_1_10  
count        4.040520e+05  
mean        -2.201967e-02  
std          1.096762e-01  
min         -7.7

In [231]:
acf_features = [
    "ret_acf_lag1_300s",
    "ret_acf_lag5_300s",
    "ret_acf_lag10_300s",
    "ret_acf_slope_1_10"
]

In [232]:
from scipy.stats import spearmanr

horizons = [30, 60, 300, 900, 1800]

acf_results = []

for date, day in df_1s.groupby("date"):

    for feature in acf_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            acf_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

acf_daily = pd.DataFrame(acf_results)

acf_summary = (
    acf_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

print(
    acf_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    acf_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon               30      60      300     900     1800
feature                                                   
ret_acf_lag10_300s  0.0062  0.0042 -0.0136 -0.0249 -0.0238
ret_acf_lag1_300s   0.0232  0.0290  0.0561  0.0322  0.0130
ret_acf_lag5_300s  -0.0055 -0.0065 -0.0187 -0.0024 -0.0143
ret_acf_slope_1_10  0.0116  0.0187  0.0533  0.0417  0.0252
horizon             30    60    300   900   1800
feature                                         
ret_acf_lag10_300s  0.60  0.60  0.33  0.33  0.40
ret_acf_lag1_300s   0.80  0.80  0.73  0.73  0.47
ret_acf_lag5_300s   0.47  0.47  0.33  0.47  0.60
ret_acf_slope_1_10  0.67  0.67  0.80  0.73  0.67


In [233]:
acf_col = "ret_acf_lag1_300s"
mom_col = "price_mom_30s_new"

rows = []

for date, day in df_1s.groupby("date"):

    temp = day[
        [acf_col, mom_col]
        + [f"fwd_ret_{h}s" for h in [30, 60, 300, 900, 1800]]
    ].copy()

    temp = temp.dropna(subset=[acf_col, mom_col])

    if len(temp) < 500:
        continue

    # Daily ACF regimes
    acf_low = temp[acf_col].quantile(0.30)
    acf_high = temp[acf_col].quantile(0.70)

    # Strong recent price moves
    mom_abs_threshold = temp[mom_col].abs().quantile(0.70)

    strong_move = (
        temp[mom_col].abs() >= mom_abs_threshold
    )

    # Mean-reversion-adjusted future return:
    # positive = future price moves opposite recent momentum
    for h in [30, 60, 300, 900, 1800]:

        mr_return = (
            -np.sign(temp[mom_col])
            * temp[f"fwd_ret_{h}s"]
        )

        negative_acf = mr_return[
            strong_move &
            (temp[acf_col] <= acf_low)
        ].dropna()

        positive_acf = mr_return[
            strong_move &
            (temp[acf_col] >= acf_high)
        ].dropna()

        if (
            len(negative_acf) < 30
            or len(positive_acf) < 30
        ):
            continue

        rows.append({
            "date": date,
            "horizon": h,

            "neg_acf_mr_mean": negative_acf.mean(),
            "pos_acf_mr_mean": positive_acf.mean(),

            "difference": (
                negative_acf.mean()
                - positive_acf.mean()
            ),

            "neg_acf_mr_frac": (
                negative_acf > 0
            ).mean(),

            "pos_acf_mr_frac": (
                positive_acf > 0
            ).mean(),

            "n_neg_acf": len(negative_acf),
            "n_pos_acf": len(positive_acf)
        })

acf_mr_daily = pd.DataFrame(rows)

In [234]:
acf_mr_summary = (
    acf_mr_daily
    .groupby("horizon")
    .agg(
        neg_acf_mr_mean=(
            "neg_acf_mr_mean", "mean"
        ),

        pos_acf_mr_mean=(
            "pos_acf_mr_mean", "mean"
        ),

        mean_difference=(
            "difference", "mean"
        ),

        median_difference=(
            "difference", "median"
        ),

        positive_days=(
            "difference",
            lambda x: (x > 0).mean()
        ),

        n_days=(
            "date", "nunique"
        )
    )
)

print(acf_mr_summary)

         neg_acf_mr_mean  pos_acf_mr_mean  mean_difference  median_difference  \
horizon                                                                         
30              0.000005         0.000008    -2.642593e-06      -8.311401e-07   
60              0.000007         0.000009    -1.325294e-06      -5.441748e-08   
300             0.000012         0.000007     5.054290e-06       2.548859e-06   
900             0.000007         0.000003     4.519807e-06       2.415571e-06   
1800            0.000010         0.000010    -6.027289e-07      -7.836320e-06   

         positive_days  n_days  
horizon                         
30            0.466667      15  
60            0.466667      15  
300           0.733333      15  
900           0.666667      15  
1800          0.400000      15  


In [235]:
ofi_col = "ofi_l1_sum_60s_signed_log"
acf_col = "ret_acf_lag1_300s"

rows = []

for date, day in df_1s.groupby("date"):

    temp = day[
        [ofi_col, acf_col]
        + [f"fwd_ret_{h}s" for h in [30, 60, 300, 900, 1800]]
    ].dropna(subset=[ofi_col, acf_col])

    if len(temp) < 500:
        continue

    # Strong OFI
    ofi_abs_threshold = (
        temp[ofi_col]
        .abs()
        .quantile(0.70)
    )

    strong_ofi = (
        temp[ofi_col].abs()
        >= ofi_abs_threshold
    )

    # Local ACF regimes
    acf_low = temp[acf_col].quantile(0.30)
    acf_high = temp[acf_col].quantile(0.70)

    for h in [30, 60, 300, 900, 1800]:

        continuation = (
            np.sign(temp[ofi_col])
            *
            temp[f"fwd_ret_{h}s"]
        )

        high_acf = continuation[
            strong_ofi &
            (temp[acf_col] >= acf_high)
        ].dropna()

        low_acf = continuation[
            strong_ofi &
            (temp[acf_col] <= acf_low)
        ].dropna()

        if len(high_acf) < 30 or len(low_acf) < 30:
            continue

        rows.append({
            "date": date,
            "horizon": h,

            "high_acf_cont": high_acf.mean(),
            "low_acf_cont": low_acf.mean(),

            "difference": (
                high_acf.mean()
                - low_acf.mean()
            ),

            "high_acf_cont_frac": (
                high_acf > 0
            ).mean(),

            "low_acf_cont_frac": (
                low_acf > 0
            ).mean()
        })

acf_ofi_daily = pd.DataFrame(rows)

In [236]:
print(
    acf_ofi_daily
    .groupby("horizon")
    .agg(
        high_acf_cont=("high_acf_cont", "mean"),
        low_acf_cont=("low_acf_cont", "mean"),
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=(
            "difference",
            lambda x: (x > 0).mean()
        ),
        n_days=("date", "nunique")
    )
)

         high_acf_cont  low_acf_cont  mean_difference  median_difference  \
horizon                                                                    
30            0.000004      0.000005    -9.075062e-07      -1.130496e-07   
60            0.000009      0.000005     4.295298e-06       2.017934e-06   
300           0.000016      0.000002     1.399274e-05       1.088248e-05   
900           0.000025      0.000003     2.207272e-05       1.352102e-05   
1800          0.000015      0.000014     1.032273e-06      -8.188337e-06   

         positive_days  n_days  
horizon                         
30            0.466667      15  
60            0.533333      15  
300           0.666667      15  
900           0.666667      15  
1800          0.466667      15  


In [237]:
import numpy as np
import pandas as pd

vr_window = 300
vr_min_periods = 120
eps = 1e-12

# ==========================================
# VARIANCE RATIO FEATURES
# ==========================================

for q in [5, 10, 30]:

    # q-second cumulative return
    df_1s[f"_ret_sum_{q}"] = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .transform(
            lambda x: x.rolling(
                q,
                min_periods=q
            ).sum()
        )
    )

    # rolling variance of 1-second returns
    var_1s = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .transform(
            lambda x: x.rolling(
                vr_window,
                min_periods=vr_min_periods
            ).var()
        )
    )

    # rolling variance of q-second cumulative returns
    var_q = (
        df_1s
        .groupby(["symbol", "date"])[f"_ret_sum_{q}"]
        .transform(
            lambda x: x.rolling(
                vr_window,
                min_periods=vr_min_periods
            ).var()
        )
    )

    df_1s[f"vr_{q}_300s"] = (
        var_q
        /
        (q * var_1s + eps)
    )

In [238]:
vr_features = [
    "vr_5_300s",
    "vr_10_300s",
    "vr_30_300s"
]

print(
    df_1s[vr_features].describe(
        percentiles=[
            0.01, 0.05,
            0.25, 0.50, 0.75,
            0.95, 0.99
        ]
    )
)

           vr_5_300s     vr_10_300s     vr_30_300s
count  407341.000000  407266.000000  406966.000000
mean        0.903890       0.836628       0.770823
std         0.319078       0.833413       4.888279
min         0.000000       0.000000       0.000000
1%          0.300212       0.189577       0.097719
5%          0.564958       0.408759       0.226906
25%         0.800079       0.660077       0.420748
50%         0.939098       0.838389       0.621273
75%         0.997418       0.980124       0.859170
95%         1.168939       1.229568       1.309696
99%         1.371185       1.539378       1.981848
max        80.759155     178.638705     624.457882


In [239]:
for q in [5, 10, 30]:
    df_1s[f"vr_{q}_300s_centered"] = (
        df_1s[f"vr_{q}_300s"] - 1.0
    )

In [240]:
vr_features = [
    "vr_5_300s",
    "vr_10_300s",
    "vr_30_300s",
    "vr_5_300s_centered",
    "vr_10_300s_centered",
    "vr_30_300s_centered",
]

In [241]:
print(
    df_1s[
        ["vr_5_300s", "vr_10_300s", "vr_30_300s"]
    ].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

           vr_5_300s     vr_10_300s     vr_30_300s
count  407341.000000  407266.000000  406966.000000
mean        0.903890       0.836628       0.770823
std         0.319078       0.833413       4.888279
min         0.000000       0.000000       0.000000
1%          0.300212       0.189577       0.097719
5%          0.564958       0.408759       0.226906
25%         0.800079       0.660077       0.420748
50%         0.939098       0.838389       0.621273
75%         0.997418       0.980124       0.859170
95%         1.168939       1.229568       1.309696
99%         1.371185       1.539378       1.981848
max        80.759155     178.638705     624.457882


In [243]:
from scipy.stats import spearmanr
import pandas as pd

vr_features = [
    "vr_5_300s",
    "vr_10_300s",
    "vr_30_300s",
    "vr_5_300s_centered",
    "vr_10_300s_centered",
    "vr_30_300s_centered",
]

horizons = [30, 60, 300, 900, 1800]

vr_results = []

for date, day in df_1s.groupby("date"):

    for feature in vr_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            vr_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

vr_daily = pd.DataFrame(vr_results)

vr_summary = (
    vr_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [244]:
print(
    vr_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

horizon                30      60      300     900     1800
feature                                                    
vr_10_300s           0.0089  0.0144  0.0338  0.0257 -0.0026
vr_10_300s_centered  0.0089  0.0144  0.0338  0.0257 -0.0026
vr_30_300s           0.0111  0.0147  0.0387  0.0413  0.0254
vr_30_300s_centered  0.0111  0.0147  0.0387  0.0413  0.0254
vr_5_300s            0.0140  0.0192  0.0490  0.0270 -0.0026
vr_5_300s_centered   0.0140  0.0192  0.0490  0.0270 -0.0026


In [245]:
print(
    vr_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon              30    60    300   900   1800
feature                                          
vr_10_300s           0.73  0.67  0.73  0.53  0.60
vr_10_300s_centered  0.73  0.67  0.73  0.53  0.60
vr_30_300s           0.80  0.87  0.67  0.60  0.60
vr_30_300s_centered  0.80  0.87  0.67  0.60  0.60
vr_5_300s            0.73  0.73  0.73  0.53  0.47
vr_5_300s_centered   0.73  0.73  0.73  0.53  0.47


In [246]:
print(
    df_1s[
        ["vr_5_300s", "vr_10_300s", "vr_30_300s"]
    ].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

           vr_5_300s     vr_10_300s     vr_30_300s
count  407341.000000  407266.000000  406966.000000
mean        0.903890       0.836628       0.770823
std         0.319078       0.833413       4.888279
min         0.000000       0.000000       0.000000
1%          0.300212       0.189577       0.097719
5%          0.564958       0.408759       0.226906
25%         0.800079       0.660077       0.420748
50%         0.939098       0.838389       0.621273
75%         0.997418       0.980124       0.859170
95%         1.168939       1.229568       1.309696
99%         1.371185       1.539378       1.981848
max        80.759155     178.638705     624.457882


In [247]:
for q in [5, 10, 30]:

    col = f"vr_{q}_300s"

    lo = df_1s[col].quantile(0.01)
    hi = df_1s[col].quantile(0.99)

    clipped = df_1s[col].clip(lo, hi)

    df_1s[f"log_vr_{q}_300s"] = np.log(
        clipped + 1e-12
    )

In [248]:
vr_col = "vr_30_300s"
mom_col = "price_mom_30s_new"

rows = []

for date, day in df_1s.groupby("date"):

    temp = day[
        [vr_col, mom_col]
        + [f"fwd_ret_{h}s" for h in [30, 60, 300, 900, 1800]]
    ].dropna(subset=[vr_col, mom_col])

    if len(temp) < 500:
        continue

    # Define relative VR regimes within each day
    vr_low = temp[vr_col].quantile(0.30)
    vr_high = temp[vr_col].quantile(0.70)

    # Focus on meaningful recent price moves
    mom_threshold = (
        temp[mom_col]
        .abs()
        .quantile(0.70)
    )

    strong_move = (
        temp[mom_col].abs()
        >= mom_threshold
    )

    for h in [30, 60, 300, 900, 1800]:

        # Positive value = future movement opposite recent price move
        mr_return = (
            -np.sign(temp[mom_col])
            *
            temp[f"fwd_ret_{h}s"]
        )

        low_vr = mr_return[
            strong_move &
            (temp[vr_col] <= vr_low)
        ].dropna()

        high_vr = mr_return[
            strong_move &
            (temp[vr_col] >= vr_high)
        ].dropna()

        if len(low_vr) < 30 or len(high_vr) < 30:
            continue

        rows.append({
            "date": date,
            "horizon": h,

            "low_vr_mr_mean": low_vr.mean(),
            "high_vr_mr_mean": high_vr.mean(),

            "difference": (
                low_vr.mean()
                - high_vr.mean()
            ),

            "low_vr_mr_frac": (
                low_vr > 0
            ).mean(),

            "high_vr_mr_frac": (
                high_vr > 0
            ).mean(),

            "n_low_vr": len(low_vr),
            "n_high_vr": len(high_vr)
        })

vr_mr_daily = pd.DataFrame(rows)

In [249]:
vr_mr_summary = (
    vr_mr_daily
    .groupby("horizon")
    .agg(
        low_vr_mr_mean=("low_vr_mr_mean", "mean"),
        high_vr_mr_mean=("high_vr_mr_mean", "mean"),

        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),

        positive_days=(
            "difference",
            lambda x: (x > 0).mean()
        ),

        n_days=("date", "nunique")
    )
)

print(vr_mr_summary)

         low_vr_mr_mean  high_vr_mr_mean  mean_difference  median_difference  \
horizon                                                                        
30             0.000006         0.000007    -1.538270e-07       2.516555e-06   
60             0.000008         0.000010    -2.323861e-06      -7.265669e-07   
300            0.000004         0.000003     1.077916e-06       4.026555e-06   
900            0.000001        -0.000008     9.499968e-06       5.449073e-06   
1800           0.000011        -0.000006     1.725554e-05       1.634532e-05   

         positive_days  n_days  
horizon                         
30            0.533333      15  
60            0.400000      15  
300           0.600000      15  
900           0.600000      15  
1800          0.533333      15  


In [250]:
window = 300
min_periods = 120
eps = 1e-12

# lag-1 return
df_1s["_ret_lag1_ar"] = (
    df_1s
    .groupby(["symbol", "date"])["ret_1s"]
    .shift(1)
)

def rolling_ar1_beta(g):

    x = g["_ret_lag1_ar"]
    y = g["ret_1s"]

    mean_x = x.rolling(window, min_periods=min_periods).mean()
    mean_y = y.rolling(window, min_periods=min_periods).mean()

    mean_xy = (
        (x * y)
        .rolling(window, min_periods=min_periods)
        .mean()
    )

    mean_x2 = (
        (x * x)
        .rolling(window, min_periods=min_periods)
        .mean()
    )

    cov_xy = mean_xy - mean_x * mean_y
    var_x = mean_x2 - mean_x * mean_x

    beta = cov_xy / (var_x + eps)

    return beta

df_1s["ar1_phi_300s"] = (
    df_1s
    .groupby(["symbol", "date"], group_keys=False)
    .apply(rolling_ar1_beta)
)

In [251]:
print(
    df_1s["ar1_phi_300s"].describe(
        percentiles=[
            0.01, 0.05, 0.25,
            0.50, 0.75, 0.95, 0.99
        ]
    )
)

count    407386.000000
mean         -0.030597
std           0.090334
min          -2.145051
1%           -0.362543
5%           -0.211526
25%          -0.051666
50%          -0.002146
75%           0.000000
95%           0.079411
99%           0.166709
max           0.837800
Name: ar1_phi_300s, dtype: float64


In [252]:
ar_features = [
    "ar1_phi_300s"
]

ar_results = []

for date, day in df_1s.groupby("date"):

    for feature in ar_features:

        for h in [30, 60, 300, 900, 1800]:

            target = f"fwd_ret_{h}s"

            temp = day[[feature, target]].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            ar_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

ar_daily = pd.DataFrame(ar_results)

ar_summary = (
    ar_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

print(
    ar_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    ar_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon         30      60      300     900     1800
feature                                             
ar1_phi_300s  0.0239  0.0301  0.0566  0.0321  0.0126
horizon       30    60    300   900   1800
feature                                   
ar1_phi_300s   0.8   0.8   0.8  0.73  0.47


In [253]:
print(
    df_1s[
        ["ar1_phi_300s", "ret_acf_lag1_300s"]
    ].corr(
        method="spearman"
    )
)

print(
    df_1s[
        ["ar1_phi_300s", "ret_acf_lag1_300s"]
    ].corr(
        method="pearson"
    )
)

                   ar1_phi_300s  ret_acf_lag1_300s
ar1_phi_300s           1.000000           0.999743
ret_acf_lag1_300s      0.999743           1.000000
                   ar1_phi_300s  ret_acf_lag1_300s
ar1_phi_300s           1.000000           0.997502
ret_acf_lag1_300s      0.997502           1.000000


In [254]:
# Take every 30th second for exploratory AR(5) estimation
ar_sample = df_1s.iloc[::30].copy()

print(len(df_1s), len(ar_sample))

409201 13641


In [255]:
import numpy as np
import pandas as pd

# -----------------------------------------
# 1. BUILD LAGS
# -----------------------------------------

for lag in range(1, 6):
    ar_sample[f"ret_lag_{lag}"] = (
        ar_sample
        .groupby(["symbol", "date"])["ret_1s"]
        .shift(lag)
    )

In [256]:
from numpy.linalg import lstsq

window = 300
min_periods = 150

def rolling_ar5(group):

    result = pd.DataFrame(
        index=group.index,
        columns=[
            "ar5_phi1",
            "ar5_phi2",
            "ar5_phi3",
            "ar5_phi4",
            "ar5_phi5"
        ],
        dtype=float
    )

    X_all = group[
        [
            "ret_lag_1",
            "ret_lag_2",
            "ret_lag_3",
            "ret_lag_4",
            "ret_lag_5"
        ]
    ].values

    y_all = group["ret_1s"].values

    for i in range(min_periods, len(group)):

        start = max(0, i - window + 1)

        X = X_all[start:i+1]
        y = y_all[start:i+1]

        valid = (
            np.isfinite(X).all(axis=1)
            &
            np.isfinite(y)
        )

        X = X[valid]
        y = y[valid]

        if len(y) < min_periods:
            continue

        # intercept
        X_design = np.column_stack(
            [np.ones(len(X)), X]
        )

        beta, _, _, _ = lstsq(
            X_design,
            y,
            rcond=None
        )

        result.iloc[i] = beta[1:]

    return result

In [258]:
ar5_coeffs = (
    ar_sample
    .groupby(
        ["symbol", "date"],
        group_keys=False
    )
    .apply(rolling_ar5)
)

ar_sample[
    [
        "ar5_phi1",
        "ar5_phi2",
        "ar5_phi3",
        "ar5_phi4",
        "ar5_phi5"
    ]
] = ar5_coeffs

In [259]:
print(
    ar_sample[
        [
            "ar5_phi1",
            "ar5_phi2",
            "ar5_phi3",
            "ar5_phi4",
            "ar5_phi5"
        ]
    ].describe(
        percentiles=[
            0.01, 0.05, 0.25,
            0.50, 0.75, 0.95, 0.99
        ]
    )
)

           ar5_phi1      ar5_phi2      ar5_phi3      ar5_phi4      ar5_phi5
count  11330.000000  11330.000000  1.133000e+04  11330.000000  11330.000000
mean      -0.020504     -0.020415 -2.838950e-02      0.005179      0.004950
std        0.089997      0.064681  6.396532e-02      0.055902      0.069684
min       -0.513506     -0.274465 -3.255257e-01     -0.135450     -0.203659
1%        -0.386926     -0.237537 -2.269445e-01     -0.105156     -0.182801
5%        -0.186863     -0.158909 -1.588795e-01     -0.071054     -0.076181
25%       -0.042199     -0.039832 -4.873355e-02     -0.019453     -0.025773
50%       -0.008541     -0.007670 -1.684396e-02     -0.002633     -0.004121
75%        0.008072     -0.000120 -7.082018e-10      0.013865      0.025554
95%        0.118552      0.068665  6.107807e-02      0.110711      0.132576
99%        0.190384      0.146978  1.358678e-01      0.203905      0.254697
max        0.574777      0.367270  2.526121e-01      0.360764      0.359729


In [260]:
print(
    ar_sample[
        [
            "ar5_phi1",
            "ar5_phi2",
            "ar5_phi3",
            "ar5_phi4",
            "ar5_phi5"
        ]
    ].corr().round(3)
)

          ar5_phi1  ar5_phi2  ar5_phi3  ar5_phi4  ar5_phi5
ar5_phi1     1.000     0.039     0.120     0.041    -0.011
ar5_phi2     0.039     1.000     0.075     0.167     0.114
ar5_phi3     0.120     0.075     1.000     0.073     0.056
ar5_phi4     0.041     0.167     0.073     1.000     0.182
ar5_phi5    -0.011     0.114     0.056     0.182     1.000


In [261]:
import numpy as np
import pandas as pd
from numpy.linalg import lstsq

# ==========================================
# 1. BUILD TRUE 1-SECOND LAGS ON FULL DATA
# ==========================================

for lag in range(1, 6):
    df_1s[f"ret_lag_{lag}"] = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .shift(lag)
    )

In [262]:
# ==========================================
# 2. SAMPLE ONLY THE ESTIMATION ENDPOINTS
# ==========================================

sample_idx = np.arange(
    0,
    len(df_1s),
    30
)

In [263]:
# ==========================================
# 3. CORRECT TRUE-1S ROLLING AR(5)
# ==========================================

window = 300
min_periods = 150

results = []

for idx in sample_idx:

    if idx < window:
        continue

    current_symbol = df_1s.iloc[idx]["symbol"]
    current_date = df_1s.iloc[idx]["date"]

    start = idx - window + 1

    block = df_1s.iloc[start:idx+1]

    # Make sure window does not cross day/symbol boundary
    block = block[
        (block["symbol"] == current_symbol)
        &
        (block["date"] == current_date)
    ]

    cols = [
        "ret_lag_1",
        "ret_lag_2",
        "ret_lag_3",
        "ret_lag_4",
        "ret_lag_5",
        "ret_1s"
    ]

    block = block[cols].dropna()

    if len(block) < min_periods:
        continue

    X = block[
        [
            "ret_lag_1",
            "ret_lag_2",
            "ret_lag_3",
            "ret_lag_4",
            "ret_lag_5"
        ]
    ].values

    y = block["ret_1s"].values

    X_design = np.column_stack(
        [np.ones(len(X)), X]
    )

    beta, _, _, _ = lstsq(
        X_design,
        y,
        rcond=None
    )

    results.append({
        "index": idx,
        "ar5_phi1": beta[1],
        "ar5_phi2": beta[2],
        "ar5_phi3": beta[3],
        "ar5_phi4": beta[4],
        "ar5_phi5": beta[5],
    })

ar5_sample = pd.DataFrame(results)

In [264]:
print(
    ar5_sample[
        [
            "ar5_phi1",
            "ar5_phi2",
            "ar5_phi3",
            "ar5_phi4",
            "ar5_phi5"
        ]
    ].describe(
        percentiles=[
            0.01, 0.05,
            0.25, 0.50, 0.75,
            0.95, 0.99
        ]
    )
)

           ar5_phi1      ar5_phi2      ar5_phi3      ar5_phi4      ar5_phi5
count  1.355900e+04  13559.000000  13559.000000  13559.000000  13559.000000
mean  -3.507699e-02     -0.026138     -0.025497     -0.023381     -0.024376
std    1.019677e-01      0.084852      0.077711      0.080470      0.079371
min   -8.333333e-01     -0.772839     -0.505571     -0.668088     -0.609905
1%    -4.186247e-01     -0.335519     -0.307686     -0.322498     -0.323345
5%    -2.309668e-01     -0.185344     -0.184570     -0.168116     -0.179021
25%   -5.753037e-02     -0.040943     -0.039910     -0.045360     -0.043590
50%   -3.389831e-03     -0.003814     -0.002728     -0.003390     -0.002600
75%    1.221844e-18      0.000000      0.000401      0.001629      0.002645
95%    8.277860e-02      0.065291      0.069197      0.075782      0.069287
99%    1.677842e-01      0.163404      0.135041      0.158522      0.154247
max    8.299320e-01      1.496584      0.496599      0.468981      0.320064


In [265]:
print(
    ar5_sample[
        [
            "ar5_phi1",
            "ar5_phi2",
            "ar5_phi3",
            "ar5_phi4",
            "ar5_phi5"
        ]
    ].corr().round(3)
)

          ar5_phi1  ar5_phi2  ar5_phi3  ar5_phi4  ar5_phi5
ar5_phi1     1.000     0.319     0.181     0.139    -0.014
ar5_phi2     0.319     1.000     0.150     0.145     0.020
ar5_phi3     0.181     0.150     1.000     0.084     0.039
ar5_phi4     0.139     0.145     0.084     1.000     0.000
ar5_phi5    -0.014     0.020     0.039     0.000     1.000


In [266]:
import numpy as np
import pandas as pd
from numpy.linalg import lstsq

window = 300
min_periods = 150

causal_results = []

for idx in sample_idx:

    # Need enough past history
    if idx < window + 5:
        continue

    current_symbol = df_1s.iloc[idx]["symbol"]
    current_date = df_1s.iloc[idx]["date"]

    # IMPORTANT:
    # use only data through t-1
    end = idx - 1
    start = end - window + 1

    block = df_1s.iloc[start:end + 1]

    block = block[
        (block["symbol"] == current_symbol)
        &
        (block["date"] == current_date)
    ]

    cols = [
        "ret_lag_1",
        "ret_lag_2",
        "ret_lag_3",
        "ret_lag_4",
        "ret_lag_5",
        "ret_1s"
    ]

    block = block[cols].dropna()

    if len(block) < min_periods:
        continue

    X = block[
        [
            "ret_lag_1",
            "ret_lag_2",
            "ret_lag_3",
            "ret_lag_4",
            "ret_lag_5"
        ]
    ].values

    y = block["ret_1s"].values

    X_design = np.column_stack([
        np.ones(len(X)),
        X
    ])

    beta, _, _, _ = lstsq(
        X_design,
        y,
        rcond=None
    )

    causal_results.append({
        "index": idx,
        "date": current_date,

        "ar5_phi1": beta[1],
        "ar5_phi2": beta[2],
        "ar5_phi3": beta[3],
        "ar5_phi4": beta[4],
        "ar5_phi5": beta[5],
    })

ar5_causal = pd.DataFrame(causal_results)

In [267]:
sample_data = df_1s.loc[
    ar5_causal["index"],
    [
        "ret_1s",
        "fwd_ret_30s",
        "fwd_ret_60s",
        "fwd_ret_300s",
        "fwd_ret_900s",
        "fwd_ret_1800s"
    ]
].copy()

sample_data.index = ar5_causal.index

ar5_causal = pd.concat(
    [
        ar5_causal.reset_index(drop=True),
        sample_data.reset_index(drop=True)
    ],
    axis=1
)

In [268]:
for lag in range(1, 6):

    ar5_causal[f"ret_lag_{lag}"] = (
        df_1s.loc[
            ar5_causal["index"],
            f"ret_lag_{lag}"
        ]
        .values
    )

In [269]:
ar5_causal["ar5_score"] = 0.0

for lag in range(1, 6):

    ar5_causal["ar5_score"] += (
        ar5_causal[f"ar5_phi{lag}"]
        *
        ar5_causal[f"ret_lag_{lag}"]
    )

In [270]:
from scipy.stats import spearmanr

ar_test_features = [
    "ar5_phi1",
    "ar5_phi2",
    "ar5_phi3",
    "ar5_phi4",
    "ar5_phi5",
    "ar5_score"
]

horizons = [30, 60, 300, 900, 1800]

rows = []

for date, day in ar5_causal.groupby("date"):

    for feature in ar_test_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[
                [feature, target]
            ].dropna()

            if len(temp) < 50:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            rows.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

ar5_ic_daily = pd.DataFrame(rows)

ar5_ic_summary = (
    ar5_ic_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic",
            lambda x: (x > 0).mean()
        ),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [271]:
print(
    ar5_ic_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    ar5_ic_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon      30      60      300     900     1800
feature                                          
ar5_phi1   0.0162  0.0321  0.0579  0.0458  0.0243
ar5_phi2   0.0010  0.0008  0.0012  0.0026 -0.0402
ar5_phi3   0.0065  0.0038  0.0117 -0.0204  0.0033
ar5_phi4   0.0001  0.0076  0.0205  0.0038 -0.0281
ar5_phi5  -0.0091 -0.0142 -0.0166  0.0058 -0.0111
ar5_score  0.0111 -0.0075  0.0141  0.0100  0.0057
horizon    30    60    300   900   1800
feature                                
ar5_phi1   0.80  0.87  0.80  0.73  0.60
ar5_phi2   0.40  0.47  0.53  0.67  0.53
ar5_phi3   0.73  0.60  0.60  0.53  0.47
ar5_phi4   0.47  0.60  0.73  0.60  0.53
ar5_phi5   0.33  0.33  0.40  0.53  0.47
ar5_score  0.73  0.33  0.67  0.73  0.60


In [272]:
import numpy as np
import pandas as pd

# ==========================================
# LOCAL HURST VIA VARIANCE SCALING
# ==========================================

lags = [1, 2, 5, 10, 30]
window = 900
min_periods = 450

hurst_results = []

for date, day in df_1s.groupby("date"):

    day = day.copy()

    logp = day["log_mid"].values

    idxs = day.index.values

    # evaluate every 30 seconds for exploration
    for j in range(window, len(day), 30):

        block = logp[j-window:j]

        vars_ = []

        valid_lags = []

        for lag in lags:

            diff = block[lag:] - block[:-lag]

            if len(diff) < 50:
                continue

            v = np.var(diff, ddof=1)

            if np.isfinite(v) and v > 0:
                vars_.append(v)
                valid_lags.append(lag)

        if len(vars_) < 3:
            continue

        x = np.log(valid_lags)
        y = np.log(vars_)

        slope, intercept = np.polyfit(x, y, 1)

        H = slope / 2.0

        hurst_results.append({
            "index": idxs[j],
            "date": date,
            "hurst_900s": H
        })

hurst_sample = pd.DataFrame(hurst_results)

In [273]:
print(
    hurst_sample["hurst_900s"].describe(
        percentiles=[
            0.01, 0.05,
            0.25, 0.50, 0.75,
            0.95, 0.99
        ]
    )
)

count    13191.000000
mean         0.430661
std          0.057739
min          0.049867
1%           0.278758
5%           0.332013
25%          0.396282
50%          0.435722
75%          0.467926
95%          0.516148
99%          0.554587
max          0.638444
Name: hurst_900s, dtype: float64


In [274]:
for h in [30, 60, 300, 900, 1800]:

    hurst_sample[f"fwd_ret_{h}s"] = (
        df_1s.loc[
            hurst_sample["index"],
            f"fwd_ret_{h}s"
        ].values
    )

In [275]:
from scipy.stats import spearmanr

rows = []

for date, day in hurst_sample.groupby("date"):

    for h in [30, 60, 300, 900, 1800]:

        temp = day[
            ["hurst_900s", f"fwd_ret_{h}s"]
        ].dropna()

        if len(temp) < 30:
            continue

        ic, _ = spearmanr(
            temp["hurst_900s"],
            temp[f"fwd_ret_{h}s"]
        )

        rows.append({
            "date": date,
            "horizon": h,
            "ic": ic
        })

hurst_daily = pd.DataFrame(rows)

hurst_summary = (
    hurst_daily
    .groupby("horizon")
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
)

print(hurst_summary)

          mean_ic  median_ic  positive_days  n_days
horizon                                            
30       0.011079   0.011757       0.600000      15
60       0.022048   0.030531       0.666667      15
300      0.035469  -0.011570       0.466667      15
900     -0.007012   0.010653       0.533333      15
1800     0.017217  -0.021922       0.466667      15


In [276]:
# ==========================================
# ROLLING RETURN DISTRIBUTION SHAPE
# ==========================================

for w in [300, 900]:

    df_1s[f"ret_skew_{w}s"] = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .transform(
            lambda x: x.rolling(
                w,
                min_periods=max(120, w // 3)
            ).skew()
        )
    )

    df_1s[f"ret_kurt_{w}s"] = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .transform(
            lambda x: x.rolling(
                w,
                min_periods=max(120, w // 3)
            ).kurt()
        )
    )

In [277]:
shape_features = [
    "ret_skew_300s",
    "ret_skew_900s",
    "ret_kurt_300s",
    "ret_kurt_900s"
]

In [278]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# ==========================================
# 1. ROLLING RETURN DISTRIBUTION SHAPE
# ==========================================

for w in [300, 900]:

    df_1s[f"ret_skew_{w}s"] = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .transform(
            lambda x: x.rolling(
                window=w,
                min_periods=max(120, w // 3)
            ).skew()
        )
    )

    df_1s[f"ret_kurt_{w}s"] = (
        df_1s
        .groupby(["symbol", "date"])["ret_1s"]
        .transform(
            lambda x: x.rolling(
                window=w,
                min_periods=max(120, w // 3)
            ).kurt()
        )
    )

# ==========================================
# 2. FEATURES TO TEST
# ==========================================

shape_features = [
    "ret_skew_300s",
    "ret_skew_900s",
    "ret_kurt_300s",
    "ret_kurt_900s"
]

# ==========================================
# 3. DAILY SPEARMAN IC TEST
# ==========================================

horizons = [30, 60, 300, 900, 1800]

shape_results = []

for date, day in df_1s.groupby("date"):

    for feature in shape_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[
                [feature, target]
            ].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            shape_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

shape_daily = pd.DataFrame(shape_results)

# ==========================================
# 4. SUMMARY
# ==========================================

shape_summary = (
    shape_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

# ==========================================
# 5. PRINT MEAN IC TABLE
# ==========================================

print(
    shape_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

# ==========================================
# 6. PRINT POSITIVE-DAYS TABLE
# ==========================================

print(
    shape_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon          30      60      300     900     1800
feature                                              
ret_kurt_300s -0.0085 -0.0093 -0.0082  0.0040 -0.0061
ret_kurt_900s -0.0066 -0.0052  0.0012 -0.0225  0.0026
ret_skew_300s -0.0522 -0.0563 -0.0548 -0.0396 -0.0349
ret_skew_900s -0.0389 -0.0443 -0.0380 -0.0257 -0.0418
horizon        30    60    300   900   1800
feature                                    
ret_kurt_300s  0.33  0.40  0.47  0.53  0.53
ret_kurt_900s  0.40  0.40  0.40  0.53  0.47
ret_skew_300s  0.07  0.07  0.27  0.33  0.27
ret_skew_900s  0.07  0.13  0.40  0.47  0.40


In [279]:
print(
    df_1s[shape_features].describe(
        percentiles=[
            0.01, 0.05, 0.25,
            0.50, 0.75, 0.95, 0.99
        ]
    )
)

       ret_skew_300s  ret_skew_900s  ret_kurt_300s  ret_kurt_900s
count  407401.000000  404701.000000  407401.000000  404701.000000
mean        0.302268       0.005927      62.958498      73.817799
std         4.470141       2.905376      48.336322      52.920627
min       -17.320508     -30.000000      -3.000000      -3.000000
1%        -12.185645      -8.804099       4.814433      14.730280
5%         -6.825030      -4.313309      14.487815      23.470974
25%        -1.628388      -1.352191      31.233935      40.773134
50%         0.060667      -0.021783      49.558138      59.360997
75%         1.992565       1.267524      79.099401      90.256497
95%         8.369809       4.514246     151.884401     170.443383
99%        13.905321       8.766126     271.518496     288.836370
max        17.320508      30.000000     300.000000     900.000000


In [280]:
df_1s["skew_reversal_signal_300s"] = (
    -df_1s["ret_skew_300s"]
)

df_1s["mean_reversion_signal_30s"] = (
    -df_1s["price_mom_30s_new"]
)

In [281]:
df_1s["mr_skew_agreement"] = (
    np.sign(df_1s["mean_reversion_signal_30s"])
    *
    np.sign(df_1s["skew_reversal_signal_300s"])
)

In [282]:
rows = []

for date, day in df_1s.groupby("date"):

    temp = day[
        [
            "mean_reversion_signal_30s",
            "skew_reversal_signal_300s",
            "mr_skew_agreement"
        ]
        + [f"fwd_ret_{h}s" for h in [30, 60, 300, 900, 1800]]
    ].dropna()

    if len(temp) < 500:
        continue

    # Focus on meaningful recent price moves
    mr_thr = (
        temp["mean_reversion_signal_30s"]
        .abs()
        .quantile(0.70)
    )

    strong_mr = (
        temp["mean_reversion_signal_30s"].abs()
        >= mr_thr
    )

    for h in [30, 60, 300, 900, 1800]:

        # Positive = future move in mean-reversion direction
        mr_return = (
            np.sign(temp["mean_reversion_signal_30s"])
            *
            temp[f"fwd_ret_{h}s"]
        )

        agree = mr_return[
            strong_mr &
            (temp["mr_skew_agreement"] > 0)
        ].dropna()

        disagree = mr_return[
            strong_mr &
            (temp["mr_skew_agreement"] < 0)
        ].dropna()

        if len(agree) < 30 or len(disagree) < 30:
            continue

        rows.append({
            "date": date,
            "horizon": h,
            "agree_mean": agree.mean(),
            "disagree_mean": disagree.mean(),
            "difference": agree.mean() - disagree.mean()
        })

mr_skew_daily = pd.DataFrame(rows)

print(
    mr_skew_daily
    .groupby("horizon")
    .agg(
        agree_mean=("agree_mean", "mean"),
        disagree_mean=("disagree_mean", "mean"),
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=("difference", lambda x: (x > 0).mean()),
        n_days=("date", "nunique")
    )
)

         agree_mean  disagree_mean  mean_difference  median_difference  \
horizon                                                                  
30         0.000009       0.000002         0.000006           0.000006   
60         0.000011       0.000003         0.000008           0.000010   
300        0.000006       0.000010        -0.000004           0.000005   
900       -0.000002       0.000009        -0.000011           0.000011   
1800       0.000003       0.000008        -0.000005           0.000006   

         positive_days  n_days  
horizon                         
30            0.866667      15  
60            0.866667      15  
300           0.600000      15  
900           0.600000      15  
1800          0.533333      15  


In [283]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

eps = 1e-12

# ==========================================
# 1. POSITIVE / NEGATIVE RETURN VARIATION
# ==========================================

df_1s["_ret_sq_pos"] = np.where(
    df_1s["ret_1s"] > 0,
    df_1s["ret_1s"] ** 2,
    0.0
)

df_1s["_ret_sq_neg"] = np.where(
    df_1s["ret_1s"] < 0,
    df_1s["ret_1s"] ** 2,
    0.0
)

# ==========================================
# 2. ROLLING SEMIVARIANCE FEATURES
# ==========================================

for w in [300, 900]:

    df_1s[f"up_semivar_{w}s"] = (
        df_1s
        .groupby(["symbol", "date"])["_ret_sq_pos"]
        .transform(
            lambda x: x.rolling(
                window=w,
                min_periods=max(120, w // 3)
            ).sum()
        )
    )

    df_1s[f"down_semivar_{w}s"] = (
        df_1s
        .groupby(["symbol", "date"])["_ret_sq_neg"]
        .transform(
            lambda x: x.rolling(
                window=w,
                min_periods=max(120, w // 3)
            ).sum()
        )
    )

# ==========================================
# 3. SEMIVARIANCE IMBALANCE
# ==========================================

for w in [300, 900]:

    up = df_1s[f"up_semivar_{w}s"]
    down = df_1s[f"down_semivar_{w}s"]

    df_1s[f"semivar_imbalance_{w}s"] = (
        (up - down)
        /
        (up + down + eps)
    )

In [284]:
semivar_features = [
    "semivar_imbalance_300s",
    "semivar_imbalance_900s"
]

print(
    df_1s[semivar_features].describe(
        percentiles=[
            0.01, 0.05, 0.25,
            0.50, 0.75, 0.95, 0.99
        ]
    )
)

       semivar_imbalance_300s  semivar_imbalance_900s
count           407416.000000           404716.000000
mean                 0.032350                0.011404
std                  0.367581                0.204102
min                 -0.999964               -0.999879
1%                  -0.913729               -0.496490
5%                  -0.571176               -0.302844
25%                 -0.182348               -0.109182
50%                  0.013731                0.004115
75%                  0.235258                0.127364
95%                  0.689803                0.342802
99%                  0.999001                0.584353
max                  0.999973                0.999971


In [285]:
horizons = [30, 60, 300, 900, 1800]

semivar_results = []

for date, day in df_1s.groupby("date"):

    for feature in semivar_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[
                [feature, target]
            ].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            semivar_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

semivar_daily = pd.DataFrame(semivar_results)

semivar_summary = (
    semivar_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [286]:
print(
    semivar_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    semivar_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon                   30      60      300     900     1800
feature                                                       
semivar_imbalance_300s -0.0604 -0.0634 -0.0619 -0.0350 -0.0448
semivar_imbalance_900s -0.0426 -0.0442 -0.0367 -0.0254 -0.0575
horizon                 30    60    300   900   1800
feature                                             
semivar_imbalance_300s  0.07  0.07  0.27  0.47  0.27
semivar_imbalance_900s  0.07  0.13  0.40  0.40  0.27


In [287]:
compare_features = [
    "ret_skew_300s",
    "ret_skew_900s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s"
]

print(
    df_1s[compare_features]
    .corr(method="spearman")
    .round(3)
)

print(
    df_1s[compare_features]
    .corr(method="pearson")
    .round(3)
)

                        ret_skew_300s  ret_skew_900s  semivar_imbalance_300s  \
ret_skew_300s                   1.000          0.410                   0.954   
ret_skew_900s                   0.410          1.000                   0.388   
semivar_imbalance_300s          0.954          0.388                   1.000   
semivar_imbalance_900s          0.431          0.911                   0.463   

                        semivar_imbalance_900s  
ret_skew_300s                            0.431  
ret_skew_900s                            0.911  
semivar_imbalance_300s                   0.463  
semivar_imbalance_900s                   1.000  
                        ret_skew_300s  ret_skew_900s  semivar_imbalance_300s  \
ret_skew_300s                   1.000          0.352                   0.940   
ret_skew_900s                   0.352          1.000                   0.349   
semivar_imbalance_300s          0.940          0.349                   1.000   
semivar_imbalance_900s          0.

In [288]:
compare_with_mom = [
    "price_mom_30s_new",
    "ret_skew_300s",
    "semivar_imbalance_300s"
]

print(
    df_1s[compare_with_mom]
    .corr(method="spearman")
    .round(3)
)

                        price_mom_30s_new  ret_skew_300s  \
price_mom_30s_new                   1.000          0.157   
ret_skew_300s                       0.157          1.000   
semivar_imbalance_300s              0.207          0.954   

                        semivar_imbalance_300s  
price_mom_30s_new                        0.207  
ret_skew_300s                            0.954  
semivar_imbalance_300s                   1.000  


In [289]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

eps = 1e-12

# ==========================================
# 1. BUILD COMPONENTS
# ==========================================

df_1s["_ret_abs"] = np.abs(df_1s["ret_1s"])

df_1s["_ret_abs_lag1"] = (
    df_1s
    .groupby(["symbol", "date"])["_ret_abs"]
    .shift(1)
)

df_1s["_bipower_component"] = (
    df_1s["_ret_abs"]
    *
    df_1s["_ret_abs_lag1"]
)

df_1s["_ret_sq_jump"] = (
    df_1s["ret_1s"] ** 2
)

# ==========================================
# 2. ROLLING RV / BV / JUMP RATIO
# ==========================================

for w in [300, 900]:

    # Realized variance
    df_1s[f"jump_rv_{w}s"] = (
        df_1s
        .groupby(["symbol", "date"])["_ret_sq_jump"]
        .transform(
            lambda x: x.rolling(
                window=w,
                min_periods=max(120, w // 3)
            ).sum()
        )
    )

    # Bipower variation
    df_1s[f"jump_bv_{w}s"] = (
        (np.pi / 2.0)
        *
        df_1s
        .groupby(["symbol", "date"])["_bipower_component"]
        .transform(
            lambda x: x.rolling(
                window=w,
                min_periods=max(120, w // 3)
            ).sum()
        )
    )

    # Estimated jump variation
    df_1s[f"jump_var_{w}s"] = (
        df_1s[f"jump_rv_{w}s"]
        -
        df_1s[f"jump_bv_{w}s"]
    ).clip(lower=0)

    # Fraction of variation attributable to jumps
    df_1s[f"jump_ratio_{w}s"] = (
        df_1s[f"jump_var_{w}s"]
        /
        (
            df_1s[f"jump_rv_{w}s"]
            + eps
        )
    )

In [290]:
jump_features = [
    "jump_ratio_300s",
    "jump_ratio_900s"
]

print(
    df_1s[jump_features].describe(
        percentiles=[
            0.01, 0.05, 0.25,
            0.50, 0.75, 0.95, 0.99
        ]
    )
)

       jump_ratio_300s  jump_ratio_900s
count    407386.000000    404686.000000
mean          0.857575         0.842968
std           0.192342         0.150138
min           0.000000         0.000000
1%            0.090925         0.325812
5%            0.476290         0.556001
25%           0.780781         0.769190
50%           0.936769         0.881435
75%           0.999851         0.959130
95%           0.999974         0.999968
99%           0.999988         0.999988
max           0.999996         0.999996


In [291]:
horizons = [30, 60, 300, 900, 1800]

jump_results = []

for date, day in df_1s.groupby("date"):

    for feature in jump_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[
                [feature, target]
            ].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            jump_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

jump_daily = pd.DataFrame(jump_results)

jump_summary = (
    jump_daily
    .groupby(["feature", "horizon"])
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=("ic", lambda x: (x > 0).mean()),
        n_days=("ic", "count")
    )
    .reset_index()
)

In [292]:
print(
    jump_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

print(
    jump_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

horizon            30      60      300     900     1800
feature                                                
jump_ratio_300s -0.0038 -0.0089  0.0101  0.0255  0.0090
jump_ratio_900s  0.0031  0.0059  0.0111 -0.0101 -0.0035
horizon          30    60    300   900   1800
feature                                      
jump_ratio_300s   0.4  0.40  0.53  0.53  0.47
jump_ratio_900s   0.6  0.53  0.53  0.53  0.47


In [293]:
jump_col = "jump_ratio_300s"
ofi_col = "ofi_l1_sum_60s_signed_log"

rows = []

for date, day in df_1s.groupby("date"):

    temp = day[
        [jump_col, ofi_col]
        + [f"fwd_ret_{h}s" for h in [30, 60, 300, 900, 1800]]
    ].dropna()

    if len(temp) < 500:
        continue

    # Strong OFI
    ofi_thr = temp[ofi_col].abs().quantile(0.70)
    strong_ofi = temp[ofi_col].abs() >= ofi_thr

    # Low vs high jump regime
    jump_low = temp[jump_col].quantile(0.30)
    jump_high = temp[jump_col].quantile(0.70)

    for h in [30, 60, 300, 900, 1800]:

        continuation = (
            np.sign(temp[ofi_col])
            *
            temp[f"fwd_ret_{h}s"]
        )

        high_jump = continuation[
            strong_ofi &
            (temp[jump_col] >= jump_high)
        ].dropna()

        low_jump = continuation[
            strong_ofi &
            (temp[jump_col] <= jump_low)
        ].dropna()

        if len(high_jump) < 30 or len(low_jump) < 30:
            continue

        rows.append({
            "date": date,
            "horizon": h,
            "high_jump_cont": high_jump.mean(),
            "low_jump_cont": low_jump.mean(),
            "difference": high_jump.mean() - low_jump.mean()
        })

jump_ofi_daily = pd.DataFrame(rows)

In [294]:
print(
    jump_ofi_daily
    .groupby("horizon")
    .agg(
        high_jump_cont=("high_jump_cont", "mean"),
        low_jump_cont=("low_jump_cont", "mean"),
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=(
            "difference",
            lambda x: (x > 0).mean()
        ),
        n_days=("date", "nunique")
    )
)

         high_jump_cont  low_jump_cont  mean_difference  median_difference  \
horizon                                                                      
30             0.000003       0.000005    -2.775378e-06          -0.000004   
60             0.000006       0.000006    -2.732263e-07           0.000003   
300            0.000016      -0.000001     1.767107e-05           0.000019   
900            0.000033       0.000004     2.984259e-05           0.000021   
1800           0.000038       0.000025     1.288939e-05           0.000004   

         positive_days  n_days  
horizon                         
30            0.400000      15  
60            0.666667      15  
300           0.733333      15  
900           0.866667      15  
1800          0.600000      15  


In [295]:
mom_col = "price_mom_30s_new"
jump_col = "jump_ratio_300s"

rows = []

for date, day in df_1s.groupby("date"):

    temp = day[
        [jump_col, mom_col]
        + [f"fwd_ret_{h}s" for h in [30, 60, 300, 900, 1800]]
    ].dropna()

    if len(temp) < 500:
        continue

    mom_thr = temp[mom_col].abs().quantile(0.70)
    strong_move = temp[mom_col].abs() >= mom_thr

    jump_low = temp[jump_col].quantile(0.30)
    jump_high = temp[jump_col].quantile(0.70)

    for h in [30, 60, 300, 900, 1800]:

        mr_return = (
            -np.sign(temp[mom_col])
            *
            temp[f"fwd_ret_{h}s"]
        )

        high_jump = mr_return[
            strong_move &
            (temp[jump_col] >= jump_high)
        ].dropna()

        low_jump = mr_return[
            strong_move &
            (temp[jump_col] <= jump_low)
        ].dropna()

        if len(high_jump) < 30 or len(low_jump) < 30:
            continue

        rows.append({
            "date": date,
            "horizon": h,
            "high_jump_mr": high_jump.mean(),
            "low_jump_mr": low_jump.mean(),
            "difference": high_jump.mean() - low_jump.mean()
        })

jump_mr_daily = pd.DataFrame(rows)

In [296]:
print(
    jump_mr_daily
    .groupby("horizon")
    .agg(
        high_jump_mr=("high_jump_mr", "mean"),
        low_jump_mr=("low_jump_mr", "mean"),
        mean_difference=("difference", "mean"),
        median_difference=("difference", "median"),
        positive_days=(
            "difference",
            lambda x: (x > 0).mean()
        ),
        n_days=("date", "nunique")
    )
)

         high_jump_mr  low_jump_mr  mean_difference  median_difference  \
horizon                                                                  
30           0.000010     0.000004         0.000006           0.000004   
60           0.000011     0.000006         0.000005           0.000003   
300          0.000002     0.000013        -0.000011          -0.000002   
900         -0.000012     0.000008        -0.000020          -0.000009   
1800        -0.000009     0.000004        -0.000013          -0.000007   

         positive_days  n_days  
horizon                         
30            0.733333      15  
60            0.666667      15  
300           0.400000      15  
900           0.400000      15  
1800          0.400000      15  


In [298]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# ==========================================
# 1. RETURN SIGN
# ==========================================

df_1s["_ret_sign"] = np.sign(
    df_1s["ret_1s"]
)

df_1s.loc[
    ~np.isfinite(df_1s["ret_1s"]),
    "_ret_sign"
] = np.nan

# ==========================================
# 2. ROLLING SIGN ENTROPY FUNCTION
# ==========================================

def rolling_sign_entropy(x, window, min_periods):

    arr = x.to_numpy(dtype=float)

    out = np.full(
        len(arr),
        np.nan,
        dtype=float
    )

    for i in range(len(arr)):

        start = max(0, i - window + 1)

        block = arr[start:i+1]

        # remove NaN / inf
        block = block[np.isfinite(block)]

        if len(block) < min_periods:
            continue

        counts = np.array(
            [
                np.sum(block == -1),
                np.sum(block == 0),
                np.sum(block == 1)
            ],
            dtype=float
        )

        total = counts.sum()

        if total == 0:
            continue

        p = counts / total

        # remove zero probabilities
        p = p[p > 0]

        entropy = -np.sum(
            p * np.log(p)
        )

        # normalize by maximum entropy log(3)
        out[i] = entropy / np.log(3)

    return pd.Series(
        out,
        index=x.index
    )

# ==========================================
# 3. COMPUTE 300s / 900s ENTROPY
# ==========================================

for w in [300, 900]:

    min_periods = max(
        120,
        w // 3
    )

    df_1s[f"sign_entropy_{w}s"] = (
        df_1s
        .groupby(
            ["symbol", "date"],
            group_keys=False
        )["_ret_sign"]
        .apply(
            lambda x: rolling_sign_entropy(
                x,
                window=w,
                min_periods=min_periods
            )
        )
    )

# ==========================================
# 4. FEATURES TO TEST
# ==========================================

entropy_features = [
    "sign_entropy_300s",
    "sign_entropy_900s"
]

# ==========================================
# 5. DISTRIBUTION CHECK
# ==========================================

print("===== ENTROPY DISTRIBUTION =====")

print(
    df_1s[
        entropy_features
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

# ==========================================
# 6. DAILY SPEARMAN IC TEST
# ==========================================

horizons = [
    30,
    60,
    300,
    900,
    1800
]

entropy_results = []

for date, day in df_1s.groupby("date"):

    for feature in entropy_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[
                [feature, target]
            ].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            entropy_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

entropy_daily = pd.DataFrame(
    entropy_results
)

# ==========================================
# 7. SUMMARY TABLE
# ==========================================

entropy_summary = (
    entropy_daily
    .groupby(
        ["feature", "horizon"]
    )
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=(
            "ic",
            lambda x: (x > 0).mean()
        ),
        n_days=("ic", "count")
    )
    .reset_index()
)

# ==========================================
# 8. PRINT MEAN IC
# ==========================================

print("\n===== MEAN DAILY IC =====")

print(
    entropy_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

# ==========================================
# 9. PRINT POSITIVE DAYS
# ==========================================

print("\n===== POSITIVE DAYS =====")

print(
    entropy_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)

===== ENTROPY DISTRIBUTION =====
       sign_entropy_300s  sign_entropy_900s
count      407401.000000      404701.000000
mean            0.250677           0.259702
std             0.162986           0.142611
min            -0.000000          -0.000000
1%              0.020335           0.062578
5%              0.056769           0.092464
25%             0.140029           0.157737
50%             0.212142           0.226682
75%             0.322043           0.324969
95%             0.583714           0.540331
99%             0.823371           0.780102
max             0.999428           0.928881

===== MEAN DAILY IC =====
horizon              30      60      300     900     1800
feature                                                  
sign_entropy_300s  0.0016  0.0013 -0.0018 -0.0036  0.0089
sign_entropy_900s  0.0058  0.0027  0.0071  0.0249  0.0093

===== POSITIVE DAYS =====
horizon            30    60    300   900   1800
feature                                        
sign_entropy_

In [299]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# =========================================================
# 1. NONZERO RETURN SIGN
# =========================================================

df_1s["_nonzero_ret_sign"] = np.nan

df_1s.loc[
    df_1s["ret_1s"] > 0,
    "_nonzero_ret_sign"
] = 1.0

df_1s.loc[
    df_1s["ret_1s"] < 0,
    "_nonzero_ret_sign"
] = -1.0


# =========================================================
# 2. BINARY SIGN ENTROPY FUNCTION
# =========================================================

def rolling_binary_sign_entropy(
    x,
    window,
    min_nonzero_moves
):

    arr = x.to_numpy(dtype=float)

    out = np.full(
        len(arr),
        np.nan,
        dtype=float
    )

    for i in range(len(arr)):

        start = max(
            0,
            i - window + 1
        )

        block = arr[start:i+1]

        # Keep only actual nonzero price moves
        block = block[
            np.isfinite(block)
        ]

        if len(block) < min_nonzero_moves:
            continue

        n_pos = np.sum(block == 1)
        n_neg = np.sum(block == -1)

        total = n_pos + n_neg

        if total == 0:
            continue

        p_pos = n_pos / total
        p_neg = n_neg / total

        entropy = 0.0

        if p_pos > 0:
            entropy -= (
                p_pos
                * np.log(p_pos)
            )

        if p_neg > 0:
            entropy -= (
                p_neg
                * np.log(p_neg)
            )

        # Normalize to [0,1]
        out[i] = (
            entropy
            / np.log(2)
        )

    return pd.Series(
        out,
        index=x.index
    )


# =========================================================
# 3. COMPUTE 300s AND 900s CONDITIONAL ENTROPY
# =========================================================

# Because moves are sparse, require a minimum number
# of actual nonzero moves rather than raw seconds.

entropy_settings = {
    300: 10,
    900: 20
}

for w, min_moves in entropy_settings.items():

    df_1s[f"direction_entropy_{w}s"] = (
        df_1s
        .groupby(
            ["symbol", "date"],
            group_keys=False
        )["_nonzero_ret_sign"]
        .apply(
            lambda x: rolling_binary_sign_entropy(
                x,
                window=w,
                min_nonzero_moves=min_moves
            )
        )
    )


# =========================================================
# 4. DIRECTIONAL BIAS
# =========================================================
#
# Entropy tells us HOW ORDERLY the direction is.
#
# We should also explicitly measure WHICH DIRECTION dominates:
#
# bias = (N_up - N_down) / (N_up + N_down)
#
# -1 = all moves down
# +1 = all moves up
#  0 = balanced
# =========================================================

def rolling_direction_bias(
    x,
    window,
    min_nonzero_moves
):

    arr = x.to_numpy(dtype=float)

    out = np.full(
        len(arr),
        np.nan,
        dtype=float
    )

    for i in range(len(arr)):

        start = max(
            0,
            i - window + 1
        )

        block = arr[start:i+1]

        block = block[
            np.isfinite(block)
        ]

        if len(block) < min_nonzero_moves:
            continue

        n_pos = np.sum(block == 1)
        n_neg = np.sum(block == -1)

        total = n_pos + n_neg

        if total == 0:
            continue

        out[i] = (
            (n_pos - n_neg)
            / total
        )

    return pd.Series(
        out,
        index=x.index
    )


for w, min_moves in entropy_settings.items():

    df_1s[f"direction_bias_{w}s"] = (
        df_1s
        .groupby(
            ["symbol", "date"],
            group_keys=False
        )["_nonzero_ret_sign"]
        .apply(
            lambda x: rolling_direction_bias(
                x,
                window=w,
                min_nonzero_moves=min_moves
            )
        )
    )


# =========================================================
# 5. FEATURES TO TEST
# =========================================================

direction_features = [
    "direction_entropy_300s",
    "direction_entropy_900s",
    "direction_bias_300s",
    "direction_bias_900s"
]


# =========================================================
# 6. DISTRIBUTION CHECK
# =========================================================

print(
    "\n===== DIRECTIONAL ENTROPY / BIAS DISTRIBUTION ====="
)

print(
    df_1s[
        direction_features
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


# =========================================================
# 7. DAILY SPEARMAN IC
# =========================================================

horizons = [
    30,
    60,
    300,
    900,
    1800
]

direction_results = []

for date, day in df_1s.groupby("date"):

    for feature in direction_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[
                [feature, target]
            ].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            direction_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })


direction_daily = pd.DataFrame(
    direction_results
)


# =========================================================
# 8. SUMMARY
# =========================================================

direction_summary = (
    direction_daily
    .groupby(
        ["feature", "horizon"]
    )
    .agg(
        mean_ic=(
            "ic",
            "mean"
        ),
        median_ic=(
            "ic",
            "median"
        ),
        std_ic=(
            "ic",
            "std"
        ),
        positive_days=(
            "ic",
            lambda x: (x > 0).mean()
        ),
        n_days=(
            "ic",
            "count"
        )
    )
    .reset_index()
)


# =========================================================
# 9. PRINT MEAN IC
# =========================================================

print(
    "\n===== MEAN DAILY IC ====="
)

print(
    direction_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)


# =========================================================
# 10. PRINT POSITIVE DAYS
# =========================================================

print(
    "\n===== POSITIVE DAYS ====="
)

print(
    direction_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)


===== DIRECTIONAL ENTROPY / BIAS DISTRIBUTION =====
       direction_entropy_300s  direction_entropy_900s  direction_bias_300s  \
count           292706.000000           374387.000000        292706.000000   
mean                 0.970199                0.986617             0.022204   
std                  0.048979                0.021101             0.199362   
min                  0.353359                0.684038            -0.866667   
1%                   0.749595                0.892623            -0.466667   
5%                   0.881291                0.945660            -0.297297   
25%                  0.964079                0.983376            -0.100000   
50%                  0.988699                0.994694             0.000000   
75%                  0.997503                0.998846             0.142857   
95%                  1.000000                1.000000             0.360000   
99%                  1.000000                1.000000             0.500000   
max        

In [300]:
# =========================================================
# REDUNDANCY CHECK:
# DIRECTION BIAS / ENTROPY VS EXISTING CONTRARIAN FEATURES
# =========================================================

compare_features = [
    "price_mom_30s_new",
    "semivar_imbalance_300s",
    "direction_bias_300s",
    "direction_bias_900s",
    "direction_entropy_300s",
    "direction_entropy_900s"
]

print("\n===== SPEARMAN CORRELATION =====")

print(
    df_1s[
        compare_features
    ]
    .corr(method="spearman")
    .round(3)
)

print("\n===== PEARSON CORRELATION =====")

print(
    df_1s[
        compare_features
    ]
    .corr(method="pearson")
    .round(3)
)


===== SPEARMAN CORRELATION =====
                        price_mom_30s_new  semivar_imbalance_300s  \
price_mom_30s_new                   1.000                   0.207   
semivar_imbalance_300s              0.207                   1.000   
direction_bias_300s                 0.222                   0.391   
direction_bias_900s                 0.135                   0.212   
direction_entropy_300s             -0.029                  -0.054   
direction_entropy_900s             -0.026                  -0.019   

                        direction_bias_300s  direction_bias_900s  \
price_mom_30s_new                     0.222                0.135   
semivar_imbalance_300s                0.391                0.212   
direction_bias_300s                   1.000                0.596   
direction_bias_900s                   0.596                1.000   
direction_entropy_300s               -0.152               -0.054   
direction_entropy_900s               -0.101               -0.235   

     

In [301]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# =========================================================
# 1. NONZERO RETURN SIGN
# =========================================================

df_1s["_move_sign"] = np.nan

df_1s.loc[
    df_1s["ret_1s"] > 0,
    "_move_sign"
] = 1.0

df_1s.loc[
    df_1s["ret_1s"] < 0,
    "_move_sign"
] = -1.0


# =========================================================
# 2. ROLLING SIGN-PERSISTENCE FUNCTION
# =========================================================

def rolling_sign_persistence(
    x,
    window,
    min_moves
):

    arr = x.to_numpy(dtype=float)

    out = np.full(
        len(arr),
        np.nan,
        dtype=float
    )

    for i in range(len(arr)):

        start = max(
            0,
            i - window + 1
        )

        block = arr[start:i+1]

        # keep only actual moves
        block = block[
            np.isfinite(block)
        ]

        if len(block) < min_moves:
            continue

        if len(block) < 2:
            continue

        prev = block[:-1]
        curr = block[1:]

        same = (
            curr == prev
        )

        out[i] = same.mean()

    return pd.Series(
        out,
        index=x.index
    )


# =========================================================
# 3. COMPUTE 300s / 900s PERSISTENCE
# =========================================================

settings = {
    300: 10,
    900: 20
}

for w, min_moves in settings.items():

    df_1s[f"sign_persistence_{w}s"] = (
        df_1s
        .groupby(
            ["symbol", "date"],
            group_keys=False
        )["_move_sign"]
        .apply(
            lambda x: rolling_sign_persistence(
                x,
                window=w,
                min_moves=min_moves
            )
        )
    )


# =========================================================
# 4. CENTER AROUND RANDOM ALTERNATION POINT
# =========================================================

for w in [300, 900]:

    df_1s[f"sign_persistence_centered_{w}s"] = (
        df_1s[f"sign_persistence_{w}s"]
        - 0.5
    )


# =========================================================
# 5. FEATURES
# =========================================================

persistence_features = [
    "sign_persistence_300s",
    "sign_persistence_900s"
]


# =========================================================
# 6. DISTRIBUTION
# =========================================================

print(
    "\n===== SIGN PERSISTENCE DISTRIBUTION ====="
)

print(
    df_1s[
        persistence_features
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


# =========================================================
# 7. DAILY SPEARMAN IC
# =========================================================

horizons = [
    30,
    60,
    300,
    900,
    1800
]

persistence_results = []

for date, day in df_1s.groupby("date"):

    for feature in persistence_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[
                [feature, target]
            ].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            persistence_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })


persistence_daily = pd.DataFrame(
    persistence_results
)


# =========================================================
# 8. SUMMARY
# =========================================================

persistence_summary = (
    persistence_daily
    .groupby(
        ["feature", "horizon"]
    )
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=(
            "ic",
            lambda x: (x > 0).mean()
        ),
        n_days=("ic", "count")
    )
    .reset_index()
)


# =========================================================
# 9. PRINT MEAN IC
# =========================================================

print(
    "\n===== MEAN DAILY IC ====="
)

print(
    persistence_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)


# =========================================================
# 10. PRINT POSITIVE DAYS
# =========================================================

print(
    "\n===== POSITIVE DAYS ====="
)

print(
    persistence_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)


===== SIGN PERSISTENCE DISTRIBUTION =====
       sign_persistence_300s  sign_persistence_900s
count          292706.000000          374387.000000
mean                0.414576               0.413341
std                 0.135599               0.093272
min                 0.000000               0.095238
1%                  0.111111               0.186047
5%                  0.200000               0.259259
25%                 0.321429               0.350649
50%                 0.416667               0.414815
75%                 0.500000               0.476190
95%                 0.642857               0.565217
99%                 0.730769               0.629630
max                 0.916667               0.809524

===== MEAN DAILY IC =====
horizon                  30      60      300     900     1800
feature                                                      
sign_persistence_300s  0.0090  0.0185  0.0292  0.0186  0.0248
sign_persistence_900s  0.0063  0.0092  0.0344  0.0405  0.0422

=====

In [302]:
import numpy as np
import pandas as pd

# =========================================================
# SETTINGS
# =========================================================

persist_col = "sign_persistence_900s"
ofi_col = "ofi_l1_sum_60s_signed_log"
mom_col = "price_mom_30s_new"

horizons = [30, 60, 300, 900, 1800]

# =========================================================
# 1. OFI x SIGN PERSISTENCE
# =========================================================

ofi_rows = []

for date, day in df_1s.groupby("date"):

    temp = day[
        [persist_col, ofi_col]
        + [f"fwd_ret_{h}s" for h in horizons]
    ].dropna()

    if len(temp) < 500:
        continue

    # Strong OFI
    ofi_thr = (
        temp[ofi_col]
        .abs()
        .quantile(0.70)
    )

    strong_ofi = (
        temp[ofi_col].abs()
        >= ofi_thr
    )

    # Persistence regimes
    persist_low = (
        temp[persist_col]
        .quantile(0.30)
    )

    persist_high = (
        temp[persist_col]
        .quantile(0.70)
    )

    for h in horizons:

        continuation = (
            np.sign(temp[ofi_col])
            *
            temp[f"fwd_ret_{h}s"]
        )

        high_persist = continuation[
            strong_ofi
            &
            (temp[persist_col] >= persist_high)
        ].dropna()

        low_persist = continuation[
            strong_ofi
            &
            (temp[persist_col] <= persist_low)
        ].dropna()

        if (
            len(high_persist) < 30
            or len(low_persist) < 30
        ):
            continue

        ofi_rows.append({
            "date": date,
            "horizon": h,
            "high_persist_cont": high_persist.mean(),
            "low_persist_cont": low_persist.mean(),
            "difference": (
                high_persist.mean()
                - low_persist.mean()
            )
        })

ofi_persist_daily = pd.DataFrame(
    ofi_rows
)

print(
    "\n===== OFI x SIGN PERSISTENCE ====="
)

print(
    ofi_persist_daily
    .groupby("horizon")
    .agg(
        high_persist_cont=(
            "high_persist_cont",
            "mean"
        ),
        low_persist_cont=(
            "low_persist_cont",
            "mean"
        ),
        mean_difference=(
            "difference",
            "mean"
        ),
        median_difference=(
            "difference",
            "median"
        ),
        positive_days=(
            "difference",
            lambda x: (x > 0).mean()
        ),
        n_days=(
            "date",
            "nunique"
        )
    )
)

# =========================================================
# 2. PRICE MEAN REVERSION x SIGN PERSISTENCE
# =========================================================

mr_rows = []

for date, day in df_1s.groupby("date"):

    temp = day[
        [persist_col, mom_col]
        + [f"fwd_ret_{h}s" for h in horizons]
    ].dropna()

    if len(temp) < 500:
        continue

    # Strong recent price move
    mom_thr = (
        temp[mom_col]
        .abs()
        .quantile(0.70)
    )

    strong_move = (
        temp[mom_col].abs()
        >= mom_thr
    )

    # Persistence regimes
    persist_low = (
        temp[persist_col]
        .quantile(0.30)
    )

    persist_high = (
        temp[persist_col]
        .quantile(0.70)
    )

    for h in horizons:

        # Positive = future move opposite recent price move
        mr_return = (
            -np.sign(temp[mom_col])
            *
            temp[f"fwd_ret_{h}s"]
        )

        low_persist = mr_return[
            strong_move
            &
            (temp[persist_col] <= persist_low)
        ].dropna()

        high_persist = mr_return[
            strong_move
            &
            (temp[persist_col] >= persist_high)
        ].dropna()

        if (
            len(low_persist) < 30
            or len(high_persist) < 30
        ):
            continue

        mr_rows.append({
            "date": date,
            "horizon": h,
            "low_persist_mr": low_persist.mean(),
            "high_persist_mr": high_persist.mean(),
            "difference": (
                low_persist.mean()
                - high_persist.mean()
            )
        })

mr_persist_daily = pd.DataFrame(
    mr_rows
)

print(
    "\n===== MEAN REVERSION x SIGN PERSISTENCE ====="
)

print(
    mr_persist_daily
    .groupby("horizon")
    .agg(
        low_persist_mr=(
            "low_persist_mr",
            "mean"
        ),
        high_persist_mr=(
            "high_persist_mr",
            "mean"
        ),
        mean_difference=(
            "difference",
            "mean"
        ),
        median_difference=(
            "difference",
            "median"
        ),
        positive_days=(
            "difference",
            lambda x: (x > 0).mean()
        ),
        n_days=(
            "date",
            "nunique"
        )
    )
)


===== OFI x SIGN PERSISTENCE =====
         high_persist_cont  low_persist_cont  mean_difference  \
horizon                                                         
30                0.000006      3.763807e-06         0.000002   
60                0.000010      5.376068e-06         0.000004   
300               0.000021      1.345747e-07         0.000021   
900               0.000026      6.358357e-06         0.000020   
1800              0.000022      5.545199e-06         0.000016   

         median_difference  positive_days  n_days  
horizon                                            
30                0.000001       0.600000      15  
60                0.000002       0.533333      15  
300               0.000018       0.800000      15  
900               0.000008       0.600000      15  
1800             -0.000001       0.466667      15  

===== MEAN REVERSION x SIGN PERSISTENCE =====
         low_persist_mr  high_persist_mr  mean_difference  median_difference  \
horizon          

In [303]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

eps = 1e-12

# =========================================================
# 1. ROLLING HIGH / LOW FEATURES
# =========================================================

for w in [300, 900]:

    rolling_high = (
        df_1s
        .groupby(["symbol", "date"])["mid"]
        .transform(
            lambda x: x.rolling(
                window=w,
                min_periods=max(120, w // 3)
            ).max()
        )
    )

    rolling_low = (
        df_1s
        .groupby(["symbol", "date"])["mid"]
        .transform(
            lambda x: x.rolling(
                window=w,
                min_periods=max(120, w // 3)
            ).min()
        )
    )

    # -----------------------------------------
    # Distance below recent high
    # positive number when current price is below high
    # -----------------------------------------

    df_1s[f"dist_from_high_{w}s"] = (
        np.log(
            rolling_high / (df_1s["mid"] + eps)
        )
    )

    # -----------------------------------------
    # Distance above recent low
    # positive number when current price is above low
    # -----------------------------------------

    df_1s[f"dist_from_low_{w}s"] = (
        np.log(
            (df_1s["mid"] + eps)
            / (rolling_low + eps)
        )
    )

    # -----------------------------------------
    # Position inside recent range
    #
    # 0   = at rolling low
    # 0.5 = middle
    # 1   = at rolling high
    # -----------------------------------------

    df_1s[f"range_position_{w}s"] = (
        (
            df_1s["mid"]
            - rolling_low
        )
        /
        (
            rolling_high
            - rolling_low
            + eps
        )
    )

    # -----------------------------------------
    # Centered range position
    #
    # negative = lower half of range
    # positive = upper half of range
    # -----------------------------------------

    df_1s[f"range_position_centered_{w}s"] = (
        df_1s[f"range_position_{w}s"]
        - 0.5
    )

# =========================================================
# 2. FEATURES TO TEST
# =========================================================

range_features = [
    "dist_from_high_300s",
    "dist_from_low_300s",
    "range_position_300s",

    "dist_from_high_900s",
    "dist_from_low_900s",
    "range_position_900s",
]

# =========================================================
# 3. DISTRIBUTION CHECK
# =========================================================

print("\n===== RANGE / EXTREME DISTRIBUTION =====")

print(
    df_1s[
        range_features
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

# =========================================================
# 4. DAILY SPEARMAN IC
# =========================================================

horizons = [
    30,
    60,
    300,
    900,
    1800
]

range_results = []

for date, day in df_1s.groupby("date"):

    for feature in range_features:

        for h in horizons:

            target = f"fwd_ret_{h}s"

            temp = day[
                [feature, target]
            ].dropna()

            if len(temp) < 100:
                continue

            ic, _ = spearmanr(
                temp[feature],
                temp[target]
            )

            range_results.append({
                "date": date,
                "feature": feature,
                "horizon": h,
                "ic": ic
            })

range_daily = pd.DataFrame(
    range_results
)

# =========================================================
# 5. SUMMARY
# =========================================================

range_summary = (
    range_daily
    .groupby(
        ["feature", "horizon"]
    )
    .agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        std_ic=("ic", "std"),
        positive_days=(
            "ic",
            lambda x: (x > 0).mean()
        ),
        n_days=("ic", "count")
    )
    .reset_index()
)

# =========================================================
# 6. PRINT MEAN IC
# =========================================================

print(
    "\n===== MEAN DAILY IC ====="
)

print(
    range_summary.pivot(
        index="feature",
        columns="horizon",
        values="mean_ic"
    ).round(4)
)

# =========================================================
# 7. PRINT POSITIVE DAYS
# =========================================================

print(
    "\n===== POSITIVE DAYS ====="
)

print(
    range_summary.pivot(
        index="feature",
        columns="horizon",
        values="positive_days"
    ).round(2)
)


===== RANGE / EXTREME DISTRIBUTION =====
       dist_from_high_300s  dist_from_low_300s  range_position_300s  \
count         4.074160e+05       407416.000000        407416.000000   
mean          1.075981e-04            0.000111             0.523047   
std           1.624810e-04            0.000132             0.345000   
min          -1.043610e-14            0.000000             0.000000   
1%           -1.043610e-14            0.000000             0.000000   
5%           -1.032507e-14            0.000000             0.000000   
25%           2.577652e-05            0.000026             0.222222   
50%           6.495405e-05            0.000078             0.521739   
75%           1.313387e-04            0.000144             0.833333   
95%           3.506562e-04            0.000349             1.000000   
99%           7.736701e-04            0.000593             1.000000   
max           3.454459e-03            0.002081             1.000000   

       dist_from_high_900s  dist_f

In [304]:
# =========================================================
# RANGE FEATURE REDUNDANCY CHECK
# =========================================================

compare_features = [
    "price_mom_30s_new",
    "semivar_imbalance_300s",
    "range_position_300s",
    "range_position_900s",
    "dist_from_high_300s",
    "dist_from_low_300s"
]

print("\n===== SPEARMAN CORRELATION =====")

print(
    df_1s[
        compare_features
    ]
    .corr(method="spearman")
    .round(3)
)

print("\n===== PEARSON CORRELATION =====")

print(
    df_1s[
        compare_features
    ]
    .corr(method="pearson")
    .round(3)
)


===== SPEARMAN CORRELATION =====
                        price_mom_30s_new  semivar_imbalance_300s  \
price_mom_30s_new                   1.000                   0.207   
semivar_imbalance_300s              0.207                   1.000   
range_position_300s                 0.438                   0.658   
range_position_900s                 0.299                   0.585   
dist_from_high_300s                -0.352                  -0.551   
dist_from_low_300s                  0.369                   0.445   

                        range_position_300s  range_position_900s  \
price_mom_30s_new                     0.438                0.299   
semivar_imbalance_300s                0.658                0.585   
range_position_300s                   1.000                0.704   
range_position_900s                   0.704                1.000   
dist_from_high_300s                  -0.750               -0.626   
dist_from_low_300s                    0.734                0.503   

     

In [305]:
import numpy as np
import pandas as pd

from scipy.cluster.hierarchy import (
    linkage,
    dendrogram,
    fcluster
)

from scipy.spatial.distance import squareform


# =========================================================
# 1. CURRENT CANDIDATE FEATURE SET
# =========================================================

candidate_features = [

    # -----------------------------------------------------
    # OFI / ORDER FLOW
    # -----------------------------------------------------

    "ofi_l1_sum_30s_signed_log",
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",

    # -----------------------------------------------------
    # ACTIVITY / MARKET STATE
    # -----------------------------------------------------

    "ofi_activity_30s",
    "ofi_activity_60s",
    "ofi_activity_300s",

    "update_rate_30s",
    "update_rate_300s",

    "price_move_rate_30s",

    # -----------------------------------------------------
    # PRICE RESPONSE
    # -----------------------------------------------------

    "impact_lambda_1800s",
    "impact_r2_1800s",

    # change this if physical column still has old name
    "absorption_z",

    # -----------------------------------------------------
    # RESTING LIQUIDITY
    # -----------------------------------------------------

    "queue_imbalance_l1",
    "depth_imbalance_l5",

    "queue_replenishment_pressure_30s",
    "queue_replenishment_pressure_60s",

    "queue_imbalance_delta",
    "depth_imbalance_delta",

    # -----------------------------------------------------
    # SPREAD / LIQUIDITY STATE
    # -----------------------------------------------------

    "spread_rank_1800s",
    "log_spread_bps",

    # -----------------------------------------------------
    # EXECUTED FLOW
    # -----------------------------------------------------

    "volume_delta_sum_300s_sl",

    # -----------------------------------------------------
    # PRICE MEAN REVERSION
    # -----------------------------------------------------

    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",

    # -----------------------------------------------------
    # VOLATILITY
    # -----------------------------------------------------

    "log_rv_300s_new",
    "rv_ratio_60_300_new",

    # -----------------------------------------------------
    # LOCAL DEPENDENCE
    # -----------------------------------------------------

    "ret_acf_lag1_300s",
    "ret_acf_slope_1_10",

    # -----------------------------------------------------
    # VARIANCE RATIO
    # -----------------------------------------------------

    "log_vr_5_300s",
    "log_vr_10_300s",
    "log_vr_30_300s",

    # -----------------------------------------------------
    # RETURN DISTRIBUTION ASYMMETRY
    # -----------------------------------------------------

    "semivar_imbalance_300s",
    "semivar_imbalance_900s",

    # -----------------------------------------------------
    # DIRECTIONAL MOVE STRUCTURE
    # -----------------------------------------------------

    "direction_bias_300s",

    "sign_persistence_900s",

    # -----------------------------------------------------
    # JUMP REGIME
    # -----------------------------------------------------

    "jump_ratio_300s",

    # -----------------------------------------------------
    # LOCAL RANGE LOCATION
    # -----------------------------------------------------

    "range_position_300s",
    "range_position_900s",
]


# =========================================================
# 2. REMOVE FEATURES THAT MAY NOT EXIST
# =========================================================

candidate_features = [
    c
    for c in candidate_features
    if c in df_1s.columns
]

print(
    "\n===== NUMBER OF CANDIDATES ====="
)

print(
    len(candidate_features)
)

print(
    "\n===== CANDIDATE FEATURES ====="
)

for c in candidate_features:
    print(c)


# =========================================================
# 3. BUILD CLEAN FEATURE FRAME
# =========================================================

X = (
    df_1s[
        candidate_features
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# =========================================================
# 4. SPEARMAN CORRELATION
# =========================================================

spearman_corr = (
    X
    .corr(
        method="spearman",
        min_periods=1000
    )
)


print(
    "\n===== SPEARMAN CORRELATION ====="
)

print(
    spearman_corr.round(3)
)


# =========================================================
# 5. FIND HIGHLY REDUNDANT PAIRS
# =========================================================

redundant_pairs = []

for i in range(
    len(candidate_features)
):

    for j in range(
        i + 1,
        len(candidate_features)
    ):

        f1 = candidate_features[i]
        f2 = candidate_features[j]

        rho = spearman_corr.loc[
            f1,
            f2
        ]

        if (
            np.isfinite(rho)
            and abs(rho) >= 0.80
        ):

            redundant_pairs.append({
                "feature_1": f1,
                "feature_2": f2,
                "spearman": rho,
                "abs_spearman": abs(rho)
            })


redundant_pairs = (
    pd.DataFrame(
        redundant_pairs
    )
)


if len(redundant_pairs) > 0:

    redundant_pairs = (
        redundant_pairs
        .sort_values(
            "abs_spearman",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print(
        "\n===== HIGH REDUNDANCY PAIRS |rho| >= 0.80 ====="
    )

    print(
        redundant_pairs[
            [
                "feature_1",
                "feature_2",
                "spearman"
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "\nNo pairs with |Spearman| >= 0.80"
    )


# =========================================================
# 6. CREATE DISTANCE MATRIX
#
# distance = 1 - |Spearman correlation|
#
# highly related features -> distance close to 0
# unrelated features      -> distance close to 1
# =========================================================

distance_matrix = (
    1.0
    -
    spearman_corr.abs()
)

# numerical cleanup
distance_matrix = (
    distance_matrix
    .clip(
        lower=0,
        upper=1
    )
)

np.fill_diagonal(
    distance_matrix.values,
    0.0
)


# =========================================================
# 7. REMOVE FEATURES WITH INVALID CORRELATION ROWS
# =========================================================

valid_features = (
    distance_matrix
    .dropna(
        axis=0,
        how="any"
    )
    .index
)

distance_matrix_clean = (
    distance_matrix
    .loc[
        valid_features,
        valid_features
    ]
)

print(
    "\n===== VALID FEATURES FOR CLUSTERING ====="
)

print(
    len(valid_features)
)


# =========================================================
# 8. HIERARCHICAL CLUSTERING
# =========================================================

condensed_distance = squareform(
    distance_matrix_clean.values,
    checks=False
)

Z = linkage(
    condensed_distance,
    method="average"
)


# =========================================================
# 9. ASSIGN CLUSTERS
#
# distance threshold 0.30 means approximately
# |Spearman| >= 0.70 tends to cluster together
# =========================================================

cluster_labels = fcluster(
    Z,
    t=0.30,
    criterion="distance"
)

cluster_table = pd.DataFrame({
    "feature": valid_features,
    "cluster": cluster_labels
})

cluster_table = (
    cluster_table
    .sort_values(
        ["cluster", "feature"]
    )
    .reset_index(drop=True)
)


print(
    "\n===== FEATURE CLUSTERS ====="
)

for cluster_id, group in cluster_table.groupby(
    "cluster"
):

    print(
        f"\nCLUSTER {cluster_id}"
    )

    for f in group["feature"]:
        print(
            "   ",
            f
        )


# =========================================================
# 10. CLUSTER SIZE SUMMARY
# =========================================================

cluster_size = (
    cluster_table
    .groupby("cluster")
    .size()
    .sort_values(
        ascending=False
    )
)

print(
    "\n===== CLUSTER SIZES ====="
)

print(
    cluster_size
)


===== NUMBER OF CANDIDATES =====
38

===== CANDIDATE FEATURES =====
ofi_l1_sum_30s_signed_log
ofi_l1_sum_60s_signed_log
ofi_rate_diff_30_300_sl
ofi_activity_30s
ofi_activity_60s
ofi_activity_300s
update_rate_30s
update_rate_300s
price_move_rate_30s
impact_lambda_1800s
impact_r2_1800s
absorption_z
queue_imbalance_l1
depth_imbalance_l5
queue_replenishment_pressure_30s
queue_replenishment_pressure_60s
queue_imbalance_delta
depth_imbalance_delta
spread_rank_1800s
log_spread_bps
volume_delta_sum_300s_sl
price_mom_10s_new
price_mom_30s_new
price_mom_60s_new
log_rv_300s_new
rv_ratio_60_300_new
ret_acf_lag1_300s
ret_acf_slope_1_10
log_vr_5_300s
log_vr_10_300s
log_vr_30_300s
semivar_imbalance_300s
semivar_imbalance_900s
direction_bias_300s
sign_persistence_900s
jump_ratio_300s
range_position_300s
range_position_900s

===== SPEARMAN CORRELATION =====
                                  ofi_l1_sum_30s_signed_log  \
ofi_l1_sum_30s_signed_log                             1.000   
ofi_l1_sum_60s_signe